# Causal Mimicry — Unified Master Notebook
### *Dose Without Cause: do base LLMs hold structured causal representations that scale with cause intensity, distinct from lexical co-occurrence?*

---
## The experimentation story (read this first)

The question lives inside the **causal-LLM-testing framework**: a behavioural number alone can't tell you whether a model *reasons* causally or just *parrots* causal-looking text (Zečević et al.; Ding & Zhang). So we test four things in sequence, and **each stage's result is what licenses the next**:

> **does the OUTPUT track causal structure → is it just word-order/frequency → does the HIDDEN state encode it → is that encoding CAUSAL (intervention)?**

The **dose/intensity axis** rides on top: we ask not just *does the model prefer the right direction* but *does its confidence scale with how strong the cause is* — and crucially whether that scaling is **representational** or merely **lexical co-occurrence** ("heavy smoking" co-occurs with "cancer"; "slight smoking" does not).

| Stage | Question | Licenses next by showing… |
|---|---|---|
| **0** Data | positive-polarity causal, reversed anti, clean forks | a controlled stimulus set |
| **1** Behavioural direction | does the continuation margin prefer the true arrow? | the base signal exists |
| **2A** Dose — graded (`base/low/moderate/high`) | does p(correct) rise with adverbial intensity? | a dose effect is present… |
| **2B** Dose — ordinal (`less/base/more`) | …and does it survive frequency-matched phrasing? | …but maybe it's lexical |
| **3A** Freq control (graded) — wordfreq unigram | is the graded effect just adverb frequency? | rule out the simplest confound |
| **3B** Freq control (more/less) — TF-IDF / co-occurrence | **is the more/less confidence change frequency-caused?** | **the hinge**: if behaviour is co-occurrence-confounded, only internal+causal evidence can establish dose-as-structure |
| **4** Hidden probes | is direction linearly decodable, above TF-IDF + random? | structure exists internally |
| **5** Word-order baseline + gap test | **spurious-vs-anti-spurious: is direction decodable *above* a pure word-order floor?** | direction ≠ word-order artifact |
| **6** Probe–output alignment | strong hidden + weak output + low r = the dissociation | encoded-but-not-said |
| **7** Scale | does the picture hold across 12 base models (410M→32B)? | robustness, not a single-model artifact |
| **8** Dose geometry | do intensity-induced hidden shifts carry causal signal? | dose is structural, held-out |
| **9** Interventions | steering → full patching → cause-token patching | is the dose representation *causally active*? |

**Why 3B and 5 are load-bearing.** 3B tells you whether the *behavioural* dose is co-occurrence (likely) — which is exactly why the title is "Dose *Without* Cause" and why we go internal. 5 is the control that stops a reviewer saying "your direction result is just word order" — forks reversed have no causal content, so if direction decodes *above* the fork-reversal floor, word-order can't explain it.

**House rules.** All CIs are 95% **group-bootstrap over `pair_id`** (relation groups, never folds). Every figure is saved per-model, **date-stamped**, in one consistent style. One scorer, one CI module, one plotting style — so all 12 models are directly comparable.

Run one model at a time (set `MODEL_KEY` in §3, restart between models). Interventions (§9) run in the same kernel before cleanup.


## §0 · Setup + house style

In [ ]:
!pip -q install transformers torch accelerate bitsandbytes scikit-learn scipy matplotlib seaborn pandas wordfreq
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import gc, os, json, re, glob
from collections import Counter
from datetime import datetime

RNG = np.random.default_rng(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
RUN_DATE = datetime.now().strftime('%Y-%m-%d')

# house style: one palette, consistent rcParams, every saved figure date-stamped
PALETTE = {'causal':'#2563eb','anti-causal':'#f59e0b','spurious':'#16a34a','fork_spurious':'#16a34a'}
CAT_COLOR = PALETTE; CAT_COLOR_S = {'causal':'#2563eb','anti-causal':'#f59e0b','fork_spurious':'#16a34a'}
GAP_COLOR = '#7c3aed'
plt.rcParams.update({'figure.dpi':110,'savefig.dpi':150,'axes.spines.top':False,'axes.spines.right':False,
                     'axes.grid':True,'grid.alpha':0.25,'font.size':11,'legend.frameon':True})

# stamp model + date on EVERY saved figure (covers both plt.savefig and fig.savefig)
# guarded so re-running this cell can't double-wrap (which caused infinite recursion)
if not getattr(matplotlib.figure.Figure.savefig, '_is_stamped', False):
    _FIG_SAVE = matplotlib.figure.Figure.savefig
    def _fig_save_stamped(self, fname, *a, **k):
        try: self.text(0.995,0.005,f'{globals().get("MODEL_KEY","?")} · {RUN_DATE}',ha='right',va='bottom',fontsize=7,color='#999')
        except Exception: pass
        return _FIG_SAVE(self, fname, *a, **k)
    _fig_save_stamped._is_stamped = True
    matplotlib.figure.Figure.savefig = _fig_save_stamped
print(f'device={DEVICE} | run date={RUN_DATE}')

## §1 · Statistics module (group-bootstrap over relation IDs — used everywhere)

In [ ]:
def bootstrap_ci_mean(values, n_boot=10000, ci=95):
    v=np.asarray(values,float)
    if len(v)<2: return (v.mean() if len(v) else np.nan),np.nan,np.nan
    b=np.array([RNG.choice(v,len(v),replace=True).mean() for _ in range(n_boot)])
    lo,hi=np.percentile(b,[(100-ci)/2,100-(100-ci)/2]); return v.mean(),lo,hi

def grouped_bootstrap_ci(df, value_col, group_col, n_boot=5000, ci=95):
    groups=df[group_col].unique(); gmap={g:df[df[group_col]==g][value_col].values for g in groups}
    b=np.array([np.concatenate([gmap[g] for g in RNG.choice(groups,len(groups),replace=True)]).mean() for _ in range(n_boot)])
    lo,hi=np.percentile(b,[(100-ci)/2,100-(100-ci)/2]); return df[value_col].mean(),lo,hi

def group_bootstrap_accuracy(item_correct, item_groups, n_boot=5000, ci=95):
    """Accuracy CI by resampling RELATION GROUPS."""
    item_correct=np.asarray(item_correct,float); item_groups=np.asarray(item_groups); groups=np.unique(item_groups)
    gmap={g:item_correct[item_groups==g] for g in groups}
    boots=np.array([np.concatenate([gmap[g] for g in RNG.choice(groups,len(groups),replace=True)]).mean() for _ in range(n_boot)])
    lo,hi=np.percentile(boots,[(100-ci)/2,100-(100-ci)/2]); return item_correct.mean(),lo,hi,boots

def unpaired_gap_distribution(boots_a, boots_b, ci=95):
    """Honest gap from two INDEPENDENT group-bootstrap distributions."""
    n=min(len(boots_a),len(boots_b)); diff=boots_a[:n]-RNG.permutation(boots_b[:n])
    lo,hi=np.percentile(diff,[(100-ci)/2,100-(100-ci)/2]); return diff.mean(),lo,hi,(diff<=0).mean()

def pearson_boot(x,y,n_boot=5000):
    x,y=np.asarray(x),np.asarray(y); r0,_=stats.pearsonr(x,y); idx=np.arange(len(x))
    b=np.array([stats.pearsonr(x[s],y[s])[0] for s in [RNG.choice(idx,len(idx),replace=True) for _ in range(n_boot)]])
    return r0,*np.percentile(b,[2.5,97.5])

print('stats ready: bootstrap_ci_mean · grouped_bootstrap_ci · group_bootstrap_accuracy · unpaired_gap_distribution · pearson_boot')


## §2 · Data — polarity-locked loader (**edit the two paths**)
Builds `SAMPLE` (causal **positive-polarity only** + reversed anti-causal + clean forks) and
`SPURIOUS_REVERSAL_SAMPLE` (forks in original + reversed order = the §5 word-order control).
Both dose scales (graded + more/less) are attached here.


In [ ]:
# ╔══ EDIT THESE TWO PATHS ══╗
CAUSAL_PATH   = '/content/causal_400.json'
SPURIOUS_PATH = '/content/spurious_forks_300.json'
# ╚══════════════════════════╝
causal_raw   = json.load(open(CAUSAL_PATH))['relations']
spurious_raw = json.load(open(SPURIOUS_PATH))['relations']
print(f'loaded {len(causal_raw)} causal, {len(spurious_raw)} spurious')

# ---- polarity is AUTHORITATIVE from the dataset (do NOT re-derive) ----
for r in causal_raw:
    assert 'polarity' in r, f'{r["id"]} missing polarity field'
causal_pos = [r for r in causal_raw if r['polarity'] == 'positive']
causal_neg = [r for r in causal_raw if r['polarity'] == 'negative']
print(f'polarity (from dataset): {len(causal_pos)} positive, {len(causal_neg)} negative')

# ---- graded intensity (Dose-A) + ordinal (Dose-B) accessors ----
LEVELS = ['base','low','moderate','high']
ORD_LEVELS = ['less','base','more']
def x_for(it, level):
    if level=='base': return it['x_text']
    return it.get(f'x_{level}', it['x_text'])
def x_ordinal(it, level):
    x=it['x_text']
    return x if level=='base' else (f'less {x}' if level=='less' else f'more {x}')

# ---- MAIN SAMPLE: POSITIVE-polarity causal only (300) + reversed anti + forks ----
# Stages 1-9 are designed for positive-polarity dose; negatives are held separately
# for the polarity-dissociation test so they don't corrupt the main dose curves.
SAMPLE={'relations':[]}
for r in causal_pos:
    SAMPLE['relations'].append({'id':r['id'],'pair_id':r['pair_id'],'category':'causal',
        'x_text':r['x_text'],'y_text':r['y_text'],'true_direction':'subject->object','polarity':'positive',
        'x_low':r.get('x_low'),'x_moderate':r.get('x_moderate'),'x_high':r.get('x_high')})
for r in causal_pos:
    SAMPLE['relations'].append({'id':f'anti_{r["id"]}','pair_id':f'anti_{r["pair_id"]}','category':'anti-causal',
        'x_text':r['y_text'],'y_text':r['x_text'],'true_direction':'object->subject','polarity':'positive',
        'x_low':f'slight {r["y_text"]}','x_moderate':f'moderate {r["y_text"]}','x_high':f'strong {r["y_text"]}'})
for r in spurious_raw:
    SAMPLE['relations'].append({'id':r['id'],'pair_id':r['pair_id'],'category':'spurious',
        'x_text':r['x_text'],'y_text':r['y_text'],'true_direction':'none','polarity':'none',
        'x_low':f'low {r["x_text"]}','x_moderate':f'moderate {r["x_text"]}','x_high':f'high {r["x_text"]}',
        'confounder_z':r.get('confounder_z'),'keep_for_headline':r.get('keep_for_headline',True)})
items=SAMPLE['relations']

# ---- NEGATIVE-polarity set, held for the polarity-dissociation test (Stage 3C / extension) ----
# experiment-ready: same schema + graded levels; for these, increasing X should DECREASE
# confidence in "X causes Y" (Δp(more) < 0) — the sign-flip that proves causal dose vs salience.
CAUSAL_NEG_ITEMS = [{'id':r['id'],'pair_id':r['pair_id'],'category':'causal','polarity':'negative',
    'x_text':r['x_text'],'y_text':r['y_text'],
    'x_low':r.get('x_low'),'x_moderate':r.get('x_moderate'),'x_high':r.get('x_high')} for r in causal_neg]

# ---- word-order control: forks original + reversed ----
SPURIOUS_REVERSAL_SAMPLE={'relations':[]}
for r in spurious_raw:
    pid=r['pair_id']
    SPURIOUS_REVERSAL_SAMPLE['relations'] += [
        {'id':f'{pid}_orig','pair_id':pid,'category':'spurious','x_text':r['x_text'],'y_text':r['y_text']},
        {'id':f'{pid}_rev','pair_id':pid,'category':'anti_spurious','x_text':r['y_text'],'y_text':r['x_text']}]
REV=SPURIOUS_REVERSAL_SAMPLE['relations']
print('SAMPLE (positive-polarity main set):',len(items),'|',dict(Counter(it['category'] for it in items)))
print(f'  -> using {len(causal_pos)} positive causal for Stages 1-9')
print(f'  -> {len(CAUSAL_NEG_ITEMS)} negative causal held for polarity-dissociation')
print('REV:',len(REV))

## §3 · Model config — size-ordered sweep (set one, restart between models)
12 base models, 410M→32B. NF4 auto for ≥24B. Gemma-2 is **gated** (needs `HF_TOKEN` + accepted license) and forced to **eager attention** (soft-capping is wrong under sdpa/flash for exact log-probs).


In [ ]:
# size-ordered: key -> (hf_id, params_B)
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, glob

MODELS = {
    'pythia-410m':            ('EleutherAI/pythia-410m',                 0.41),
    'olmo-1b':                ('allenai/OLMo-1B-0724-hf',                1.18),
    'pythia-6.9b':            ('EleutherAI/pythia-6.9b',                 6.9),
    'olmo-7b':                ('allenai/OLMo-7B-0724-hf',                6.9),
    'qwen-7b-base':           ('Qwen/Qwen2.5-7B',                        7.6),
    'gemma-2-9b-base':        ('google/gemma-2-9b',                      9.2),   # gated, eager
    'pythia-12b':             ('EleutherAI/pythia-12b',                  11.8),
    'mistral-nemo-12b-base':  ('mistralai/Mistral-Nemo-Base-2407',       12.2),
    'olmo-2-13b':             ('allenai/OLMo-2-1124-13B',                13.7),
    'qwen-14b-base':          ('Qwen/Qwen2.5-14B',                       14.8),
    'mistral-small-24b-base': ('mistralai/Mistral-Small-24B-Base-2501',  24.0),
    'gemma-2-27b-base':       ('google/gemma-2-27b',                     27.2),  # gated, eager, NF4
    'qwen-32b-base':          ('Qwen/Qwen2.5-32B',                       32.8),  # NF4
}

RUN_ORDER = list(MODELS.keys())   # already size-ordered

MODEL_KEY = 'olmo-2-13b'  # ← change + restart runtime per model
MODEL_ID, MODEL_SIZE_B = MODELS[MODEL_KEY]

USE_NF4   = MODEL_SIZE_B >= 24
IS_GEMMA  = 'gemma' in MODEL_KEY.lower()
DATA_VERSION = 'v4'

OUT = '/content/drive/MyDrive/causal_mimicry'  # Drive, persists across reruns
OUTDIR = f'{OUT}/{MODEL_KEY}'
os.makedirs(OUTDIR, exist_ok=True)

def tag(name):
    return f'{OUTDIR}/{MODEL_KEY}_{DATA_VERSION}_{RUN_DATE}_{name}'

# one-time rescue: pull anything already saved on ephemeral /content into Drive
_old = glob.glob('/content/causal_mimicry/**/*', recursive=True)
if _old:
    shutil.copytree('/content/causal_mimicry', OUT, dirs_exist_ok=True)
    print(f'↻ rescued {len([p for p in _old if p.endswith(".png")])} figures from /content into Drive')

print(f'{MODEL_KEY} ({MODEL_SIZE_B}B) -> {MODEL_ID} | NF4={USE_NF4} gemma={IS_GEMMA}')
print(f'outputs -> {OUTDIR}/  (saved straight to Drive, prefixed {MODEL_KEY}_{DATA_VERSION}_{RUN_DATE}_)')

## §4 · Load model + prompt-conditioned scorer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token

_kw = dict(device_map='auto', trust_remote_code=True, output_hidden_states=True)
if IS_GEMMA: _kw['attn_implementation']='eager'          # required for correct gemma-2 logits
if USE_NF4:
    from transformers import BitsAndBytesConfig
    _kw['quantization_config']=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
else:
    _kw['torch_dtype']=torch.float16
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_kw).eval()
n_layers = model.config.num_hidden_layers
print(f'loaded {MODEL_KEY}: {n_layers} layers | dtype {next(model.parameters()).dtype}')

NEUTRAL_PROMPT = 'Consider the variables: {x} and {y}. The relation between them is that '

@torch.no_grad()
def cond_logprob(prompt, continuation):
    """logP(continuation|prompt), length-normalised over continuation tokens."""
    p_ids=tok(prompt,return_tensors='pt').input_ids.to(model.device)
    full =tok(prompt+continuation,return_tensors='pt').input_ids.to(model.device)
    n=p_ids.shape[1]
    if full.shape[1]<=n: return float('-inf')
    lp=F.log_softmax(model(full).logits,dim=-1)
    return lp[0,n-1:-1,:].gather(-1,full[0,n:].unsqueeze(-1)).squeeze(-1).mean().item()

def conts(x,y): return {'forward':f'{x} directly causes {y}','reverse':f'{y} directly causes {x}',
                        'none':f'{x} and {y} are not directly causally related'}
CAT2KEY={'causal':'forward','anti-causal':'reverse','spurious':'none'}
def score_dirs(x,y):
    pr=NEUTRAL_PROMPT.format(x=x,y=y); c=conts(x,y); ks=['forward','reverse','none']
    p=F.softmax(torch.tensor([cond_logprob(pr,c[k]) for k in ks]),0)
    return {k:p[i].item() for i,k in enumerate(ks)}

g=cond_logprob(NEUTRAL_PROMPT.format(x='heavy smoking',y='lung cancer'),'heavy smoking directly causes lung cancer')
b=cond_logprob(NEUTRAL_PROMPT.format(x='heavy smoking',y='lung cancer'),'lung cancer directly causes heavy smoking')
print(f'sanity — correct {g:.3f} vs reverse {b:.3f} (correct should be ≥ reverse)')


## §5 · Stage 1 — Behavioural direction (base) + the word-order caveat
Directional margin = logP(forward) − logP(reverse). Causal should be **positive** (prefers the true arrow);
fork-spurious should sit near **0** (no preferred direction) — the behavioural face of "word-order alone
shouldn't manufacture a direction". The decisive word-order test is the §5-probe gap test in Stage 5.


In [ ]:
rows=[]
for it in items:
    if it['category'] not in CAT2KEY: continue
    p=score_dirs(it['x_text'], it['y_text'])
    rows.append({'model':MODEL_KEY,'id':it['id'],'pair_id':it['pair_id'],'category':it['category'],
                 'p_forward':p['forward'],'p_reverse':p['reverse'],'p_none':p['none'],
                 'margin_dir':np.log(max(p['forward'],1e-9))-np.log(max(p['reverse'],1e-9)),
                 'p_correct':p[CAT2KEY[it['category']]]})
base_df=pd.DataFrame(rows); base_df.to_csv(tag('behavioural_direction.csv'),index=False)

fig,ax=plt.subplots(figsize=(6.5,4.4))
xs=range(3); cats=['causal','anti-causal','spurious']
ms=[];los=[];his=[]
for cat in cats:
    m,lo,hi=grouped_bootstrap_ci(base_df[base_df.category==cat],'margin_dir','pair_id'); ms+=[m];los+=[lo];his+=[hi]
ax.bar(xs,ms,yerr=[np.array(ms)-np.array(los),np.array(his)-np.array(ms)],capsize=4,
       color=[PALETTE[c] for c in cats])
ax.axhline(0,ls='--',color='grey',alpha=.6); ax.set_xticks(list(xs)); ax.set_xticklabels(cats)
ax.set_ylabel('directional margin logP(fwd)−logP(rev) [95% CI]')
ax.set_title(f'{MODEL_KEY} · Stage 1: behavioural direction')
plt.tight_layout(); plt.savefig(tag('stage1_direction.png')); plt.show()
print('causal margin > 0 and ≫ fork margin (≈0) ⇒ output tracks the true arrow, not first-mention order.')


## §6 · Stage 2A — Dose, graded adverbial (`base/low/moderate/high`)
Natural intensity phrasing (slight/moderate/heavy). Does p(correct) rise with intensity?
Carries a frequency confound (heavy ≫ slight) — controlled in §8 (Stage 3A).


In [ ]:
from wordfreq import zipf_frequency
def get_freq_for_item(it, level):
    if level=='base': return 0.0
    return zipf_frequency(x_for(it,level).strip().split()[0].lower(),'en')

rows=[]; tot=len([it for it in items if it['category'] in CAT2KEY])*len(LEVELS); done=0
for it in items:
    if it['category'] not in CAT2KEY: continue
    ck=CAT2KEY[it['category']]
    for lvl in LEVELS:
        x=x_for(it,lvl); p=score_dirs(x,it['y_text'])
        rows.append({'model':MODEL_KEY,'id':it['id'],'pair_id':it['pair_id'],'category':it['category'],'level':lvl,
                     'p_correct':p[ck],'p_forward':p['forward'],'p_reverse':p['reverse'],'p_none':p['none'],
                     'word_freq':get_freq_for_item(it,lvl)})
        done+=1
        if done%150==0: print(f'  {done}/{tot}')
gdf=pd.DataFrame(rows); gdf.to_csv(tag('dose_graded.csv'),index=False)
print('graded dose rows:',len(gdf))


In [ ]:
order={'base':0,'low':1,'moderate':2,'high':3}
fig,ax=plt.subplots(figsize=(7,4.6))
for cat in ['causal','anti-causal','spurious']:
    sub=gdf[gdf.category==cat]; ms=[];los=[];his=[]
    for lvl in LEVELS:
        m,lo,hi=grouped_bootstrap_ci(sub[sub.level==lvl],'p_correct','pair_id'); ms+=[m];los+=[lo];his+=[hi]
    xs=[order[l] for l in LEVELS]
    ax.plot(xs,ms,'o-',color=PALETTE[cat],lw=2.2,ms=7,label=cat); ax.fill_between(xs,los,his,color=PALETTE[cat],alpha=.15)
ax.axhline(1/3,ls='--',color='grey',alpha=.6,label='chance')
ax.set_xticks(list(order.values())); ax.set_xticklabels(LEVELS); ax.set_xlabel('cause intensity')
ax.set_ylabel('p(correct) [95% CI]'); ax.set_title(f'{MODEL_KEY} · Stage 2A: graded dose'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(tag('stage2A_graded_dose.png')); plt.show()


## §7 · Stage 2B — Dose, ordinal `less/base/more` (frequency-matched)
Designed to neutralise the rare-bigram confound. Two reads: (1) does p(correct) scale; (2) is more-vs-less
**symmetric** (grading = causal cue) or **asymmetric** (direction matters)?


In [ ]:
rows=[]
for it in items:
    if it['category'] not in CAT2KEY: continue
    ck=CAT2KEY[it['category']]
    for lvl in ORD_LEVELS:
        xt=x_ordinal(it,lvl); p=score_dirs(xt,it['y_text'])
        rows.append({'model':MODEL_KEY,'id':it['id'],'pair_id':it['pair_id'],'category':it['category'],
                     'polarity':it.get('polarity','none'),'level':lvl,
                     'p_forward':p['forward'],'p_reverse':p['reverse'],'p_none':p['none'],'p_correct':p[ck]})
mldf=pd.DataFrame(rows); mldf.to_csv(tag('dose_moreless.csv'),index=False)

def level_index(level,polarity):
    i={'less':0,'base':1,'more':2}[level]; return 2-i if polarity=='negative' else i
mldf['lvl_idx']=[level_index(l,p) for l,p in zip(mldf.level,mldf.polarity)]
fig,ax=plt.subplots(figsize=(7,4.6))
for cat in ['causal','anti-causal','spurious']:
    sub=mldf[mldf.category==cat]; ms=[];los=[];his=[]
    for i in range(3):
        m,lo,hi=grouped_bootstrap_ci(sub[sub.lvl_idx==i],'p_correct','pair_id'); ms+=[m];los+=[lo];his+=[hi]
    ax.plot(range(3),ms,'o-',color=PALETTE[cat],lw=2.2,ms=7,label=cat); ax.fill_between(range(3),los,his,color=PALETTE[cat],alpha=.15)
ax.axhline(1/3,ls='--',color='grey',alpha=.6,label='chance'); ax.set_xticks(range(3)); ax.set_xticklabels(['less→','base','more→'])
ax.set_xlabel('ordinal cause level (frequency-matched)'); ax.set_ylabel('p(correct) [95% CI]')
ax.set_title(f'{MODEL_KEY} · Stage 2B: less/base/more dose'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(tag('stage2B_moreless_dose.png')); plt.show()


In [ ]:
# symmetry: Δmore vs Δless, and causal-vs-spurious specificity
piv=mldf.pivot_table(index=['id','pair_id','category','polarity'],columns='level',values='p_correct').reset_index()
piv['delta_more']=piv['more']-piv['base']; piv['delta_less']=piv['less']-piv['base']
piv.to_csv(tag('dose_moreless_deltas.csv'),index=False)
print('=== grading shift vs bare base (Δmore, Δless) ===')
for cat in ['causal','anti-causal','spurious']:
    dm=piv[piv.category==cat]['delta_more'].dropna().values; dl=piv[piv.category==cat]['delta_less'].dropna().values
    mm,mlo,mhi=bootstrap_ci_mean(dm); ml,llo,lhi=bootstrap_ci_mean(dl)
    print(f'  {cat:12s} Δmore={mm:+.4f} [{mlo:+.4f},{mhi:+.4f}] | Δless={ml:+.4f} [{llo:+.4f},{lhi:+.4f}]')
cz=piv[piv.category=='causal']['delta_more'].dropna().values; sp=piv[piv.category=='spurious']['delta_more'].dropna().values
if len(cz)>2 and len(sp)>2:
    u,p=stats.mannwhitneyu(cz,sp,alternative='greater'); print(f'\ncausal Δmore > spurious Δmore: p={p:.4f} AUC={u/(len(cz)*len(sp)):.3f}')
czl=piv[piv.category=='causal']['delta_less'].dropna().values
if len(cz)>2 and len(czl)>2:
    _,psym=stats.mannwhitneyu(cz,czl,alternative='two-sided'); print(f'causal symmetry Δmore vs Δless: p={psym:.4f}  (NS=symmetric V / SIG=asymmetric slope)')


## §8 · Stage 3A — Frequency control for **graded** dose (wordfreq unigram)
Does the graded effect survive removing the per-item adverb frequency? Residualise p(correct) on word_freq;
audit whether frequency drives p(correct) equally across categories (artifact) or not (signal).


In [ ]:
from sklearn.linear_model import LinearRegression
gdf_r=gdf.copy()
lr=LinearRegression().fit(gdf_r[['word_freq']].values, gdf_r['p_correct'].values)
gdf_r['p_resid']=gdf_r['p_correct'].values - lr.predict(gdf_r[['word_freq']].values)

fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,col,ylab,ttl in [(axes[0],'p_correct','mean p(correct) [95% CI]','raw'),
                        (axes[1],'p_resid','residualised p(correct) [95% CI]','frequency-residualised')]:
    for cat in ['causal','anti-causal','spurious']:
        sub=gdf_r[gdf_r.category==cat]; ms=[];los=[];his=[]
        for lvl in LEVELS:
            m,lo,hi=grouped_bootstrap_ci(sub[sub.level==lvl],col,'pair_id'); ms+=[m];los+=[lo];his+=[hi]
        xs=[order[l] for l in LEVELS]
        ax.plot(xs,ms,'o-',color=PALETTE[cat],lw=2.2,ms=7,label=cat); ax.fill_between(xs,los,his,color=PALETTE[cat],alpha=.15)
    ax.axhline(1/3 if col=='p_correct' else 0,ls='--',color='grey',alpha=.6)
    ax.set_xticks(list(order.values())); ax.set_xticklabels(LEVELS); ax.set_xlabel('cause intensity'); ax.set_ylabel(ylab)
    ax.set_title(f'{MODEL_KEY} · Stage 3A · {ttl}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(tag('stage3A_graded_freqcontrol.png')); plt.show()

# confound audit: slope of p(correct) on freq, per category
fig,axes=plt.subplots(1,3,figsize=(14,4.3))
for ax,cat in zip(axes,['causal','anti-causal','spurious']):
    s=gdf[(gdf.category==cat)&(gdf.level!='base')]
    if len(s)<5: ax.set_title(f'{cat} — n/a'); continue
    r,p=stats.pearsonr(s['word_freq'],s['p_correct']); mf,bf=np.polyfit(s['word_freq'],s['p_correct'],1)
    xs=np.linspace(s['word_freq'].min(),s['word_freq'].max(),50)
    ax.scatter(s['word_freq'],s['p_correct'],s=18,alpha=.3,color=PALETTE[cat]); ax.plot(xs,mf*xs+bf,color=PALETTE[cat],lw=2)
    ax.set_title(f'{cat}\nr={r:.3f}, p={p:.4f}'); ax.set_xlabel('Zipf freq of intensity word'); ax.set_ylabel('p(correct)')
plt.suptitle(f'{MODEL_KEY} · Stage 3A: does adverb frequency drive p(correct)?',y=1.02); plt.tight_layout()
plt.savefig(tag('stage3A_freq_audit.png')); plt.show()
print('r(causal)≈r(anti)≈r(spur)>0 ⇒ fluency artifact. r(causal)>0 but r(anti)≤0 ⇒ freq cannot explain the differential.')


## §9 · Stage 3B — **the hinge**: is the more/less confidence change frequency-caused?
*Purpose (your spec):* check whether the confidence shift from adding **more**/**less** is driven by **lexical
co-occurrence**, not a magnitude representation. Two measures:

1. **Within-model co-occurrence likelihood** — score the *non-causal* conjunction `"{marked x} and {y}"`
   (no causal verb). If `Δ co-occurrence (more−base)` correlates with `Δ p(correct) (more−base)`, the confidence
   change is co-occurrence-driven.
2. **TF-IDF / corpus association** (optional external) — quantify marked-cause↔effect association independent
   of the model and correlate with the dose delta.

**Reading:** high correlation on causal ⇒ behavioural dose is co-occurrence-confounded → *this is what licenses
going internal (Stages 4–9)*. Low correlation ⇒ a structural component survives in behaviour itself.


In [ ]:
# 1 · within-model co-occurrence likelihood (non-causal conjunction), less/base/more
ASSOC = 'The following are often mentioned together: '
def cooc_logprob(xt, y): return cond_logprob(ASSOC, f'{xt} and {y}')   # length-normalised logP, no causal verb

crows=[]
for it in items:
    if it['category'] not in CAT2KEY: continue
    for lvl in ORD_LEVELS:
        xt=x_ordinal(it,lvl)
        crows.append({'id':it['id'],'pair_id':it['pair_id'],'category':it['category'],'level':lvl,
                      'cooc':cooc_logprob(xt,it['y_text'])})
cdf=pd.DataFrame(crows)
cpiv=cdf.pivot_table(index=['id','pair_id','category'],columns='level',values='cooc').reset_index()
cpiv['delta_cooc_more']=cpiv['more']-cpiv['base']; cpiv['delta_cooc_less']=cpiv['less']-cpiv['base']

# join to the Dose-B p(correct) deltas (piv from §7)
J=cpiv.merge(piv[['id','category','delta_more','delta_less']],on=['id','category'],how='inner')
J.to_csv(tag('stage3B_cooccurrence.csv'),index=False)

print('=== corr( Δco-occurrence , Δp(correct) ) for the MORE shift, per category ===')
corr_rows=[]
for cat in ['causal','anti-causal','spurious']:
    d=J[J.category==cat].dropna(subset=['delta_cooc_more','delta_more'])
    if len(d)<5: continue
    r,lo,hi=pearson_boot(d['delta_cooc_more'].values, d['delta_more'].values)
    verdict='CO-OCCURRENCE-DRIVEN' if lo>0.3 else ('partial' if lo>0 else 'not co-occ-driven')
    print(f'  {cat:12s} r={r:+.3f} [{lo:+.3f},{hi:+.3f}]  -> {verdict}')
    corr_rows.append({'model':MODEL_KEY,'category':cat,'r':r,'r_lo':lo,'r_hi':hi,'n':len(d)})
pd.DataFrame(corr_rows).to_csv(tag('stage3B_corr.csv'),index=False)


In [ ]:
# scatter: Δco-occurrence vs Δp(correct), per category (the visual of the hinge)
fig,axes=plt.subplots(1,3,figsize=(14,4.3))
for ax,cat in zip(axes,['causal','anti-causal','spurious']):
    d=J[J.category==cat].dropna(subset=['delta_cooc_more','delta_more'])
    if len(d)<5: ax.set_title(f'{cat} — n/a'); continue
    r,lo,hi=pearson_boot(d['delta_cooc_more'].values,d['delta_more'].values)
    ax.scatter(d['delta_cooc_more'],d['delta_more'],s=20,alpha=.4,color=PALETTE[cat])
    mf,bf=np.polyfit(d['delta_cooc_more'],d['delta_more'],1); xs=np.linspace(d['delta_cooc_more'].min(),d['delta_cooc_more'].max(),50)
    ax.plot(xs,mf*xs+bf,color=PALETTE[cat],lw=2); ax.axhline(0,ls='--',color='grey',alpha=.4); ax.axvline(0,ls='--',color='grey',alpha=.4)
    ax.set_xlabel('Δ co-occurrence logP (more−base)'); ax.set_ylabel('Δ p(correct) (more−base)')
    ax.set_title(f'{cat}\nr={r:+.3f} [{lo:+.3f},{hi:+.3f}]')
plt.suptitle(f'{MODEL_KEY} · Stage 3B: is the more/less confidence shift co-occurrence-driven?',y=1.02)
plt.tight_layout(); plt.savefig(tag('stage3B_cooccurrence_scatter.png')); plt.show()
print('High causal r ⇒ behavioural dose is largely co-occurrence (frequency) → go internal (Stages 4–9).')


In [ ]:
# ============================================================
# Stage 3C — Naturalness / frame-fluency control for MORE dose
# Cohesive version for this notebook: uses `items`, `score_dirs`, `CAT2KEY`, `tag`
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import pearsonr

# ------------------------------------------------------------
# Goal
# ------------------------------------------------------------
# Stage 2B showed that "more X causes Y" can suppress causal p(correct)
# relative to the bare "X causes Y" frame.
#
# This cell tests whether that is a surface-naturalness / frame-fluency artifact:
# maybe "more smoking causes cancer" is awkward, while
# "increased smoking causes cancer" or "higher levels of smoking cause cancer"
# is a more natural way to express causal dose.
#
# Good result:
#   causal p(correct) recovers under fluent MORE frames,
#   while spurious stays flat or low.
#
# Bad / fluency-driven result:
#   all categories increase whenever the sentence is more natural.

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------
needed_globals = ["items", "score_dirs", "CAT2KEY", "MODEL_KEY"]
missing = [x for x in needed_globals if x not in globals()]
if missing:
    raise ValueError(f"Missing required notebook variables/functions: {missing}")

if "tag" not in globals():
    def tag(name):
        return name

print(f"MODEL_KEY = {MODEL_KEY}")
print(f"Number of items = {len(items)}")
print("Categories:", pd.Series([it.get("category") for it in items]).value_counts().to_dict())

# ------------------------------------------------------------
# Full-sentence naturalness helper
# ------------------------------------------------------------
# Uses the model's own mean logP/token for the statement.
# Higher / less negative = more fluent under the model.

def sentence_logprob_per_token(text):
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    ids = enc["input_ids"]

    if ids.shape[1] < 2:
        return np.nan

    with torch.no_grad():
        logits = model(ids).logits[:, :-1, :]
        target = ids[:, 1:]
        lp = torch.log_softmax(logits, dim=-1)
        token_lp = lp.gather(-1, target.unsqueeze(-1)).squeeze(-1)

    return token_lp.mean().item()

# ------------------------------------------------------------
# Frame definitions
# ------------------------------------------------------------
# We compare the awkward original Stage 2B more-frame against more fluent
# ways of expressing increased dose.

def clean_x(x):
    return str(x).strip()

def clean_y(y):
    return str(y).strip()

FRAMES = {
    "bare": lambda x, y: (x, y),
    "awkward_more": lambda x, y: (f"more {x}", y),
    "increased": lambda x, y: (f"increased {x}", y),
    "higher_levels": lambda x, y: (f"higher levels of {x}", y),
    "greater_exposure": lambda x, y: (f"greater exposure to {x}", y),
}

# For naturalness, convert the same X/Y frame into a plain sentence.
def statement_from_xy(x_frame, y):
    return f"{x_frame} causes {y}."

# Optional extra risk frame, scored separately because it changes predicate wording.
# It is useful but less directly comparable to score_dirs(x, y), so we keep it out
# of the main p(correct) curve unless you want to add a custom scorer later.

# ------------------------------------------------------------
# Build sample
# ------------------------------------------------------------
STAGE3C_MAX_PER_CAT = 250

stage_items = [it for it in items if it.get("category") in ["causal", "anti-causal", "spurious"]]
rng = np.random.default_rng(42)

sampled = []
for cat in ["causal", "anti-causal", "spurious"]:
    cat_items = [it for it in stage_items if it.get("category") == cat]
    if len(cat_items) > STAGE3C_MAX_PER_CAT:
        idx = rng.choice(len(cat_items), STAGE3C_MAX_PER_CAT, replace=False)
        cat_items = [cat_items[i] for i in idx]
    sampled.extend(cat_items)

print("Stage 3C sampled:", pd.Series([it["category"] for it in sampled]).value_counts().to_dict())

# ------------------------------------------------------------
# Run scoring
# ------------------------------------------------------------
rows = []

for it in tqdm(sampled, desc="Stage 3C frame naturalness"):
    cat = it["category"]
    ck = CAT2KEY[cat]

    x0 = clean_x(it["x_text"])
    y0 = clean_y(it["y_text"])

    for frame_name, frame_fn in FRAMES.items():
        x_frame, y_frame = frame_fn(x0, y0)

        # Behavioural score with your existing 3-way direction scorer
        p = score_dirs(x_frame, y_frame)

        # Naturalness of the statement frame itself
        statement = statement_from_xy(x_frame, y_frame)
        nat_lp = sentence_logprob_per_token(statement)

        rows.append({
            "model": MODEL_KEY,
            "id": it.get("id"),
            "pair_id": it.get("pair_id", it.get("id")),
            "category": cat,
            "frame": frame_name,
            "x_text": x0,
            "y_text": y0,
            "x_frame": x_frame,
            "statement": statement,
            "naturalness_lp": nat_lp,
            "p_forward": p["forward"],
            "p_reverse": p["reverse"],
            "p_none": p["none"],
            "p_correct": p[ck],
        })

stage3c = pd.DataFrame(rows)
stage3c.to_csv(tag("stage3C_naturalness_frame_control.csv"), index=False)
print("Saved:", tag("stage3C_naturalness_frame_control.csv"))
display(stage3c.head())

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
def se(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

summary = (
    stage3c
    .groupby(["category", "frame"])
    .agg(
        mean_p_correct=("p_correct", "mean"),
        se_p_correct=("p_correct", se),
        mean_naturalness=("naturalness_lp", "mean"),
        se_naturalness=("naturalness_lp", se),
        n=("p_correct", "size"),
    )
    .reset_index()
)

display(summary)

# ------------------------------------------------------------
# Plot 1 — p(correct) by frame
# ------------------------------------------------------------
frame_order = list(FRAMES.keys())
cat_order = ["causal", "anti-causal", "spurious"]

fig, ax = plt.subplots(figsize=(9, 4.8))

for cat in cat_order:
    tmp = summary[summary.category == cat].set_index("frame").reindex(frame_order)
    xs = np.arange(len(frame_order))
    ys = tmp["mean_p_correct"].values
    ses = tmp["se_p_correct"].values

    ax.plot(xs, ys, "o-", color=PALETTE[cat], lw=2.4, ms=7, label=cat)
    ax.fill_between(xs, ys - 1.96 * ses, ys + 1.96 * ses, color=PALETTE[cat], alpha=0.15)

ax.axhline(1/3, ls="--", color="grey", alpha=0.6, label="chance")
ax.set_xticks(np.arange(len(frame_order)))
ax.set_xticklabels(frame_order, rotation=20, ha="right")
ax.set_ylabel("mean p(correct) [95% CI]")
ax.set_xlabel("MORE-dose frame")
ax.set_title(f"{MODEL_KEY} · Stage 3C: does fluent MORE framing recover causal confidence?")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(tag("stage3C_pcorrect_by_frame.png"), dpi=180)
plt.show()

# ------------------------------------------------------------
# Plot 2 — statement naturalness by frame
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4.8))

for cat in cat_order:
    tmp = summary[summary.category == cat].set_index("frame").reindex(frame_order)
    xs = np.arange(len(frame_order))
    ys = tmp["mean_naturalness"].values
    ses = tmp["se_naturalness"].values

    ax.plot(xs, ys, "o-", color=PALETTE[cat], lw=2.4, ms=7, label=cat)
    ax.fill_between(xs, ys - 1.96 * ses, ys + 1.96 * ses, color=PALETTE[cat], alpha=0.15)

ax.set_xticks(np.arange(len(frame_order)))
ax.set_xticklabels(frame_order, rotation=20, ha="right")
ax.set_ylabel("mean sentence logP/token\nhigher = more natural")
ax.set_xlabel("MORE-dose frame")
ax.set_title(f"{MODEL_KEY} · Stage 3C: frame naturalness")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(tag("stage3C_naturalness_by_frame.png"), dpi=180)
plt.show()

# ------------------------------------------------------------
# Plot 3 — naturalness vs p(correct)
# ------------------------------------------------------------
corr_rows = []

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, cat in zip(axes, cat_order):
    tmp = stage3c[stage3c.category == cat].dropna(subset=["naturalness_lp", "p_correct"])

    ax.scatter(tmp["naturalness_lp"], tmp["p_correct"], alpha=0.35, s=18, color=PALETTE[cat])

    if len(tmp) > 3:
        r, pval = pearsonr(tmp["naturalness_lp"], tmp["p_correct"])
        r2 = r ** 2
        m, b = np.polyfit(tmp["naturalness_lp"], tmp["p_correct"], 1)
        xs = np.linspace(tmp["naturalness_lp"].min(), tmp["naturalness_lp"].max(), 100)
        ax.plot(xs, m * xs + b, lw=2, color=PALETTE[cat])
        ax.set_title(f"{cat}\nr={r:+.3f}, R²={r2:.3f}")
        corr_rows.append({
            "category": cat,
            "r_naturalness_vs_pcorrect": r,
            "r2": r2,
            "p_value": pval,
            "n": len(tmp),
        })
    else:
        ax.set_title(cat)

    ax.set_xlabel("sentence naturalness logP/token")
    ax.grid(alpha=0.2)

axes[0].set_ylabel("p(correct)")
plt.suptitle(f"{MODEL_KEY} · Stage 3C: is p(correct) naturalness-driven?", y=1.05)
plt.tight_layout()
plt.savefig(tag("stage3C_naturalness_vs_pcorrect.png"), dpi=180, bbox_inches="tight")
plt.show()

corr_df = pd.DataFrame(corr_rows)
display(corr_df)
corr_df.to_csv(tag("stage3C_naturalness_correlations.csv"), index=False)

# ------------------------------------------------------------
# Frame contrasts vs awkward_more and vs bare
# ------------------------------------------------------------
wide = stage3c.pivot_table(
    index=["id", "pair_id", "category", "x_text", "y_text"],
    columns="frame",
    values=["p_correct", "naturalness_lp"],
    aggfunc="mean"
).reset_index()

wide.columns = [
    "_".join([str(z) for z in col if str(z) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in wide.columns
]

contrast_rows = []

for cat in cat_order:
    tmp = wide[wide.category == cat].copy()

    for frame in frame_order:
        if frame == "bare":
            continue

        pc = f"p_correct_{frame}"
        nat = f"naturalness_lp_{frame}"

        if pc in tmp.columns and "p_correct_bare" in tmp.columns:
            d_pc_bare = tmp[pc] - tmp["p_correct_bare"]
            d_nat_bare = tmp[nat] - tmp["naturalness_lp_bare"]

            row = {
                "category": cat,
                "frame": frame,
                "delta_pcorrect_vs_bare": d_pc_bare.mean(),
                "se_delta_pcorrect_vs_bare": se(d_pc_bare),
                "delta_naturalness_vs_bare": d_nat_bare.mean(),
                "se_delta_naturalness_vs_bare": se(d_nat_bare),
                "n": d_pc_bare.notna().sum(),
            }

            if "p_correct_awkward_more" in tmp.columns and frame != "awkward_more":
                d_pc_awk = tmp[pc] - tmp["p_correct_awkward_more"]
                d_nat_awk = tmp[nat] - tmp["naturalness_lp_awkward_more"]
                row["delta_pcorrect_vs_awkward_more"] = d_pc_awk.mean()
                row["se_delta_pcorrect_vs_awkward_more"] = se(d_pc_awk)
                row["delta_naturalness_vs_awkward_more"] = d_nat_awk.mean()
                row["se_delta_naturalness_vs_awkward_more"] = se(d_nat_awk)

            contrast_rows.append(row)

contrast_df = pd.DataFrame(contrast_rows)
display(contrast_df)
contrast_df.to_csv(tag("stage3C_frame_contrasts.csv"), index=False)

print("\nInterpretation guide:")
print("1. If awkward_more is less natural and lowers causal p(correct), Stage 2B MORE was surface-form fragile.")
print("2. If increased/higher_levels/greater_exposure recover causal p(correct), while spurious stays flat/low, that supports causal-dose structure beyond awkward wording.")
print("3. If naturalness strongly predicts p(correct) for all categories, the result is fluency-driven.")
print("4. If causal recovers selectively and spurious does not, that is the good result.")

In [ ]:
# ============================================================
# Stage 3D — Naturalness / frame-fluency control for ABSOLUTE DOSE
# Cohesive version for this notebook: uses `items`, `score_dirs`, `CAT2KEY`, `tag`
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from scipy.stats import pearsonr

# ------------------------------------------------------------
# Goal
# ------------------------------------------------------------
# Stage 2A/3A tests absolute dose:
#   bare/base, low/light, moderate, high/heavy
#
# But we need to know whether those curves are affected by naturalness:
# maybe "light smoking causes cancer" or "moderate smoking causes cancer"
# has different fluency than "heavy smoking causes cancer".
#
# This cell asks:
#   1. How natural is each intensity frame?
#   2. Does p(correct) track naturalness?
#   3. Does causal p(correct) rise with high intensity even after accounting for naturalness?
#
# Good result:
#   causal remains dose-sensitive, while spurious/anti-causal do not simply rise
#   with frame naturalness.

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------
needed_globals = ["items", "score_dirs", "CAT2KEY", "MODEL_KEY"]
missing = [x for x in needed_globals if x not in globals()]
if missing:
    raise ValueError(f"Missing required notebook variables/functions: {missing}")

if "tag" not in globals():
    def tag(name):
        return name

if "PALETTE" not in globals():
    PALETTE = {
        "causal": "#2E6BE6",
        "anti-causal": "#F39C12",
        "spurious": "#20A65A",
    }

print(f"MODEL_KEY = {MODEL_KEY}")
print(f"Number of items = {len(items)}")
print("Categories:", pd.Series([it.get("category") for it in items]).value_counts().to_dict())

# ------------------------------------------------------------
# Full-sentence naturalness helper
# ------------------------------------------------------------
# Higher / less negative = more fluent/natural under the model.

def sentence_logprob_per_token(text):
    enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    ids = enc["input_ids"]

    if ids.shape[1] < 2:
        return np.nan

    with torch.no_grad():
        logits = model(ids).logits[:, :-1, :]
        target = ids[:, 1:]
        lp = torch.log_softmax(logits, dim=-1)
        token_lp = lp.gather(-1, target.unsqueeze(-1)).squeeze(-1)

    return token_lp.mean().item()

def se(x):
    x = pd.Series(x).dropna()
    return x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan

def clean_x(x):
    return str(x).strip()

def clean_y(y):
    return str(y).strip()

# ------------------------------------------------------------
# Absolute-dose frames
# ------------------------------------------------------------
# These are generic; later you can improve them with domain-specific adjectives.
# Main direct test:
#   bare: X causes Y
#   low: light X causes Y
#   moderate: moderate X causes Y
#   high: heavy X causes Y
#
# Safer alternative forms:
#   low_level: low levels of X cause Y
#   moderate_level: moderate levels of X cause Y
#   high_level: high levels of X cause Y
#
# Why include both?
#   "light smoking" is natural, but "light poverty" / "light pollution" may be weird.
#   "low levels of X" is often more general.

ABS_DOSE_FRAMES = {
    "bare": lambda x, y: (x, y),
    "low_adj": lambda x, y: (f"light {x}", y),
    "moderate_adj": lambda x, y: (f"moderate {x}", y),
    "high_adj": lambda x, y: (f"heavy {x}", y),
    "low_level": lambda x, y: (f"low levels of {x}", y),
    "moderate_level": lambda x, y: (f"moderate levels of {x}", y),
    "high_level": lambda x, y: (f"high levels of {x}", y),
}

FRAME_GROUPS = {
    "adjective": ["bare", "low_adj", "moderate_adj", "high_adj"],
    "level": ["bare", "low_level", "moderate_level", "high_level"],
}

def statement_from_xy(x_frame, y):
    return f"{x_frame} causes {y}."

# Numeric dose index for monotonic tests.
DOSE_INDEX = {
    "bare": 0,
    "low_adj": 1,
    "moderate_adj": 2,
    "high_adj": 3,
    "low_level": 1,
    "moderate_level": 2,
    "high_level": 3,
}

# ------------------------------------------------------------
# Build sample
# ------------------------------------------------------------
STAGE3D_MAX_PER_CAT = 250

stage_items = [it for it in items if it.get("category") in ["causal", "anti-causal", "spurious"]]
rng = np.random.default_rng(42)

sampled = []
for cat in ["causal", "anti-causal", "spurious"]:
    cat_items = [it for it in stage_items if it.get("category") == cat]
    if len(cat_items) > STAGE3D_MAX_PER_CAT:
        idx = rng.choice(len(cat_items), STAGE3D_MAX_PER_CAT, replace=False)
        cat_items = [cat_items[i] for i in idx]
    sampled.extend(cat_items)

print("Stage 3D sampled:", pd.Series([it["category"] for it in sampled]).value_counts().to_dict())

# ------------------------------------------------------------
# Run scoring
# ------------------------------------------------------------
rows = []

for it in tqdm(sampled, desc="Stage 3D absolute-dose naturalness"):
    cat = it["category"]
    ck = CAT2KEY[cat]

    x0 = clean_x(it["x_text"])
    y0 = clean_y(it["y_text"])

    for frame_name, frame_fn in ABS_DOSE_FRAMES.items():
        x_frame, y_frame = frame_fn(x0, y0)

        # Behavioural score with your existing 3-way direction scorer
        p = score_dirs(x_frame, y_frame)

        # Naturalness of the statement frame
        statement = statement_from_xy(x_frame, y_frame)
        nat_lp = sentence_logprob_per_token(statement)

        rows.append({
            "model": MODEL_KEY,
            "id": it.get("id"),
            "pair_id": it.get("pair_id", it.get("id")),
            "category": cat,
            "frame": frame_name,
            "dose_index": DOSE_INDEX[frame_name],
            "x_text": x0,
            "y_text": y0,
            "x_frame": x_frame,
            "statement": statement,
            "naturalness_lp": nat_lp,
            "p_forward": p["forward"],
            "p_reverse": p["reverse"],
            "p_none": p["none"],
            "p_correct": p[ck],
        })

stage3d = pd.DataFrame(rows)
stage3d.to_csv(tag("stage3D_absolute_dose_naturalness_control.csv"), index=False)
print("Saved:", tag("stage3D_absolute_dose_naturalness_control.csv"))
display(stage3d.head())

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------
summary = (
    stage3d
    .groupby(["category", "frame"])
    .agg(
        mean_p_correct=("p_correct", "mean"),
        se_p_correct=("p_correct", se),
        mean_naturalness=("naturalness_lp", "mean"),
        se_naturalness=("naturalness_lp", se),
        n=("p_correct", "size"),
    )
    .reset_index()
)

display(summary)

# ------------------------------------------------------------
# Plot helper
# ------------------------------------------------------------
cat_order = ["causal", "anti-causal", "spurious"]

def plot_group(group_name, frame_order):
    # Plot p(correct)
    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    for cat in cat_order:
        tmp = summary[summary.category == cat].set_index("frame").reindex(frame_order)
        xs = np.arange(len(frame_order))
        ys = tmp["mean_p_correct"].values
        ses = tmp["se_p_correct"].values

        ax.plot(xs, ys, "o-", color=PALETTE[cat], lw=2.4, ms=7, label=cat)
        ax.fill_between(xs, ys - 1.96 * ses, ys + 1.96 * ses, color=PALETTE[cat], alpha=0.15)

    ax.axhline(1/3, ls="--", color="grey", alpha=0.6, label="chance")
    ax.set_xticks(np.arange(len(frame_order)))
    ax.set_xticklabels(frame_order, rotation=20, ha="right")
    ax.set_ylabel("mean p(correct) [95% CI]")
    ax.set_xlabel("absolute dose frame")
    ax.set_title(f"{MODEL_KEY} · Stage 3D: p(correct) under {group_name} dose frames")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(tag(f"stage3D_pcorrect_{group_name}_frames.png"), dpi=180)
    plt.show()

    # Plot naturalness
    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    for cat in cat_order:
        tmp = summary[summary.category == cat].set_index("frame").reindex(frame_order)
        xs = np.arange(len(frame_order))
        ys = tmp["mean_naturalness"].values
        ses = tmp["se_naturalness"].values

        ax.plot(xs, ys, "o-", color=PALETTE[cat], lw=2.4, ms=7, label=cat)
        ax.fill_between(xs, ys - 1.96 * ses, ys + 1.96 * ses, color=PALETTE[cat], alpha=0.15)

    ax.set_xticks(np.arange(len(frame_order)))
    ax.set_xticklabels(frame_order, rotation=20, ha="right")
    ax.set_ylabel("mean sentence logP/token\nhigher = more natural")
    ax.set_xlabel("absolute dose frame")
    ax.set_title(f"{MODEL_KEY} · Stage 3D: naturalness under {group_name} dose frames")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(tag(f"stage3D_naturalness_{group_name}_frames.png"), dpi=180)
    plt.show()

for group_name, frame_order in FRAME_GROUPS.items():
    plot_group(group_name, frame_order)

# ------------------------------------------------------------
# Naturalness vs p(correct), within category
# ------------------------------------------------------------
corr_rows = []

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)

for ax, cat in zip(axes, cat_order):
    tmp = stage3d[stage3d.category == cat].dropna(subset=["naturalness_lp", "p_correct"])

    ax.scatter(tmp["naturalness_lp"], tmp["p_correct"], alpha=0.30, s=16, color=PALETTE[cat])

    if len(tmp) > 3:
        r, pval = pearsonr(tmp["naturalness_lp"], tmp["p_correct"])
        r2 = r ** 2
        m, b = np.polyfit(tmp["naturalness_lp"], tmp["p_correct"], 1)
        xs = np.linspace(tmp["naturalness_lp"].min(), tmp["naturalness_lp"].max(), 100)
        ax.plot(xs, m * xs + b, lw=2, color=PALETTE[cat])
        ax.set_title(f"{cat}\nr={r:+.3f}, R²={r2:.3f}")
        corr_rows.append({
            "category": cat,
            "r_naturalness_vs_pcorrect": r,
            "r2": r2,
            "p_value": pval,
            "n": len(tmp),
        })
    else:
        ax.set_title(cat)

    ax.set_xlabel("sentence naturalness logP/token")
    ax.grid(alpha=0.2)

axes[0].set_ylabel("p(correct)")
plt.suptitle(f"{MODEL_KEY} · Stage 3D: is absolute-dose p(correct) naturalness-driven?", y=1.05)
plt.tight_layout()
plt.savefig(tag("stage3D_naturalness_vs_pcorrect.png"), dpi=180, bbox_inches="tight")
plt.show()

corr_df = pd.DataFrame(corr_rows)
display(corr_df)
corr_df.to_csv(tag("stage3D_naturalness_correlations.csv"), index=False)

# ------------------------------------------------------------
# Contrasts vs bare
# ------------------------------------------------------------
wide = stage3d.pivot_table(
    index=["id", "pair_id", "category", "x_text", "y_text"],
    columns="frame",
    values=["p_correct", "naturalness_lp"],
    aggfunc="mean"
).reset_index()

wide.columns = [
    "_".join([str(z) for z in col if str(z) != ""]).strip("_")
    if isinstance(col, tuple) else col
    for col in wide.columns
]

contrast_rows = []

for cat in cat_order:
    tmp = wide[wide.category == cat].copy()

    for frame in list(ABS_DOSE_FRAMES.keys()):
        if frame == "bare":
            continue

        pc = f"p_correct_{frame}"
        nat = f"naturalness_lp_{frame}"

        if pc in tmp.columns and "p_correct_bare" in tmp.columns:
            d_pc_bare = tmp[pc] - tmp["p_correct_bare"]
            d_nat_bare = tmp[nat] - tmp["naturalness_lp_bare"]

            contrast_rows.append({
                "category": cat,
                "frame": frame,
                "delta_pcorrect_vs_bare": d_pc_bare.mean(),
                "se_delta_pcorrect_vs_bare": se(d_pc_bare),
                "delta_naturalness_vs_bare": d_nat_bare.mean(),
                "se_delta_naturalness_vs_bare": se(d_nat_bare),
                "n": d_pc_bare.notna().sum(),
            })

contrast_df = pd.DataFrame(contrast_rows)
display(contrast_df)
contrast_df.to_csv(tag("stage3D_frame_contrasts_vs_bare.csv"), index=False)

# ------------------------------------------------------------
# Naturalness-adjusted dose test
# ------------------------------------------------------------
# Simple residualisation:
# p(correct) ~ naturalness_lp within category.
# Then plot residual mean by frame.
#
# This asks:
#   after removing the linear naturalness effect,
#   does causal high-dose still differ from low/moderate/bare?

stage3d_resid = stage3d.copy()
stage3d_resid["p_correct_resid_nat"] = np.nan

for cat in cat_order:
    idx = stage3d_resid.category == cat
    tmp = stage3d_resid.loc[idx].dropna(subset=["naturalness_lp", "p_correct"])

    if len(tmp) > 3:
        x = tmp["naturalness_lp"].values
        y = tmp["p_correct"].values
        m, b = np.polyfit(x, y, 1)
        pred = m * stage3d_resid.loc[idx, "naturalness_lp"].values + b
        stage3d_resid.loc[idx, "p_correct_resid_nat"] = stage3d_resid.loc[idx, "p_correct"].values - pred

resid_summary = (
    stage3d_resid
    .groupby(["category", "frame"])
    .agg(
        mean_resid=("p_correct_resid_nat", "mean"),
        se_resid=("p_correct_resid_nat", se),
        n=("p_correct_resid_nat", "size"),
    )
    .reset_index()
)

display(resid_summary)
resid_summary.to_csv(tag("stage3D_naturalness_residual_summary.csv"), index=False)

def plot_resid_group(group_name, frame_order):
    fig, ax = plt.subplots(figsize=(8.5, 4.8))

    for cat in cat_order:
        tmp = resid_summary[resid_summary.category == cat].set_index("frame").reindex(frame_order)
        xs = np.arange(len(frame_order))
        ys = tmp["mean_resid"].values
        ses = tmp["se_resid"].values

        ax.plot(xs, ys, "o-", color=PALETTE[cat], lw=2.4, ms=7, label=cat)
        ax.fill_between(xs, ys - 1.96 * ses, ys + 1.96 * ses, color=PALETTE[cat], alpha=0.15)

    ax.axhline(0, ls="--", color="grey", alpha=0.6)
    ax.set_xticks(np.arange(len(frame_order)))
    ax.set_xticklabels(frame_order, rotation=20, ha="right")
    ax.set_ylabel("naturalness-residualised p(correct)")
    ax.set_xlabel("absolute dose frame")
    ax.set_title(f"{MODEL_KEY} · Stage 3D: residual p(correct) under {group_name} frames")
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(tag(f"stage3D_residual_pcorrect_{group_name}_frames.png"), dpi=180)
    plt.show()

for group_name, frame_order in FRAME_GROUPS.items():
    plot_resid_group(group_name, frame_order)

stage3d_resid.to_csv(tag("stage3D_absolute_dose_naturalness_residualised.csv"), index=False)

print("\nInterpretation guide:")
print("1. If high-dose causal p(correct) remains high after residualising naturalness, absolute-dose evidence is not just fluency.")
print("2. If causal follows naturalness tightly but spurious/anti do not, the output readout is category-specific naturalness-sensitive.")
print("3. If all categories rise with naturalness, then the absolute-dose behaviour is mostly fluency-driven.")
print("4. If high/heavy is less natural but causal p(correct) still rises, that is especially strong evidence for causal-dose structure.")

## §10 · Stage 4 — Hidden-state probes (is direction linearly decodable, above TF-IDF + random?)
Neutral answer-free prompt; final-token states at selected layers. Per-layer probes with **TF-IDF baseline**
(pairs are lexically near-identical for causal-vs-anti) and **random-label** control; GroupKFold by `pair_id`;
relation-level bootstrap CIs. *Model stays loaded for Stage 9.*


In [ ]:
selected_layers=sorted(set([1,n_layers//4,n_layers//2,3*n_layers//4,n_layers]))
print('layers:',selected_layers,'of',n_layers)
def hs_prompt(x,y): return f'Consider the variables: {x} and {y}. The relation between them is'

@torch.no_grad()
def hidden_for(text):
    ids=tok(text,return_tensors='pt').to(model.device); hs=model(**ids).hidden_states
    last=ids['attention_mask'][0].sum().item()-1
    return np.stack([hs[L][0,last,:].float().cpu().numpy() for L in selected_layers])

def extract(lst,fields):
    X=[];meta=[]
    for it in lst:
        x=it['x_text'];y=it['y_text']; X.append(hidden_for(hs_prompt(x,y)).astype(np.float16))
        meta.append({f:it.get(f) for f in fields}|{'x_text':x,'y_text':y})
    return np.stack(X),pd.DataFrame(meta)

Xmain,mmain=extract(items,['id','pair_id','category','true_direction'])
mmain['text']=mmain.x_text.astype(str)+' '+mmain.y_text.astype(str)
np.savez_compressed(tag('hidden_main.npz'),X=Xmain,layers=np.array(selected_layers)); mmain.to_csv(tag('meta_main.csv'),index=False)
Xrev,mrev=extract(REV,['id','pair_id','category']); mrev['text']=mrev.x_text.astype(str)+' '+mrev.y_text.astype(str)
np.savez_compressed(tag('hidden_rev.npz'),X=Xrev,layers=np.array(selected_layers)); mrev.to_csv(tag('meta_rev.csv'),index=False)
print('hidden main:',Xmain.shape,'| reversal:',Xrev.shape,'| model kept loaded for Stage 9')


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold, cross_val_score

def oof_predictions(X,y,groups,shuffle=False):
    if shuffle: y=RNG.permutation(y)
    k=min(5,len(np.unique(groups))); gkf=GroupKFold(k); correct=np.zeros(len(y))
    for tr,te in gkf.split(X,y,groups=groups):
        nc=max(2,min(128,len(tr)-1,X.shape[1]))
        clf=make_pipeline(StandardScaler(),PCA(n_components=nc,random_state=0),LogisticRegression(max_iter=1000))
        clf.fit(X[tr],y[tr]); correct[te]=(clf.predict(X[te])==y[te]).astype(float)
    return correct
def tfidf_acc(texts,y,groups):
    k=min(5,len(np.unique(groups)))
    return cross_val_score(make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=1),LogisticRegression(max_iter=1000)),
                           texts,y,groups=groups,cv=GroupKFold(k),scoring='accuracy',n_jobs=-1).mean()

d=np.load(tag('hidden_main.npz'),allow_pickle=True); Xall=d['X']; layer_idx=list(d['layers'])
M=pd.read_csv(tag('meta_main.csv')); cats=M['category'].values; ids=M['pair_id'].values; texts=M['text'].values
TASKS={'causal_vs_anti':['causal','anti-causal'],'causal_vs_spurious':['causal','spurious'],
       'anti_vs_spurious':['anti-causal','spurious'],'direct_vs_spurious':'direct','3way':['causal','anti-causal','spurious']}
def task_sel_lab(t):
    spec=TASKS[t]
    if spec=='direct': m=np.isin(cats,['causal','anti-causal','spurious']); return m,np.where(cats=='spurious','spurious','direct')
    return np.isin(cats,spec),cats

probe_rows=[]; boot_store={}
for task in TASKS:
    mask,lab=task_sel_lab(task); g=ids[mask]; y=lab[mask]; tx=texts[mask]
    chance=1/len(np.unique(y)); tb=tfidf_acc(tx,y,g)
    for li,L in enumerate(layer_idx):
        X=Xall[mask,li,:].astype(np.float32)
        corr=oof_predictions(X,y,g); rcorr=oof_predictions(X,y,g,shuffle=True)
        acc,lo,hi,boots=group_bootstrap_accuracy(corr,g); racc,_,_,_=group_bootstrap_accuracy(rcorr,g)
        boot_store[(task,int(L))]=boots
        probe_rows.append({'model':MODEL_KEY,'task':task,'layer':int(L),'n':int(mask.sum()),'chance':round(chance,3),
                           'acc':round(acc,3),'acc_lo':round(lo,3),'acc_hi':round(hi,3),'tfidf':round(tb,3),'random':round(racc,3)})
    print(f'  {task:20s} done')
PM=pd.DataFrame(probe_rows); PM.to_csv(tag('probes.csv'),index=False)

# plot: 5-panel separability
fig,axes=plt.subplots(1,len(TASKS),figsize=(4*len(TASKS),4.2),sharey=True)
for ax,task in zip(axes,TASKS):
    t=PM[PM.task==task].sort_values('layer')
    ax.plot(t.layer,t.acc,'o-',color='#2563eb',lw=2,label='hidden probe'); ax.fill_between(t.layer,t.acc_lo,t.acc_hi,color='#2563eb',alpha=.15)
    ax.plot(t.layer,t.tfidf,'s--',color='#f59e0b',label='TF-IDF'); ax.plot(t.layer,t.random,'^:',color='#16a34a',label='random')
    ax.axhline(t.chance.iloc[0],ls='--',color='grey',alpha=.5); ax.set_title(task,fontsize=10); ax.set_xlabel('layer')
axes[0].set_ylabel('accuracy [95% CI over relations]'); axes[0].legend(fontsize=8)
plt.suptitle(f'{MODEL_KEY} · Stage 4: hidden-state separability',y=1.03); plt.tight_layout()
plt.savefig(tag('stage4_probes.png')); plt.show()


## §11 · Stage 5 — Word-order baseline + gap test (**the "word order isn't enough" control**)
`spurious-vs-anti-spurious` (forks reversed — pure word-order, no causal content) is the floor. The gap test asks:
is **causal-vs-anti** decodable *significantly above* that floor? Honest **unpaired group-bootstrap** of the two
independent relation-level distributions. If the gap CI excludes 0, direction ≠ word-order artifact.


In [ ]:
dr=np.load(tag('hidden_rev.npz'),allow_pickle=True); Xr=dr['X']; Mr=pd.read_csv(tag('meta_rev.csv'))
yr=Mr['category'].values; gr=Mr['pair_id'].values; txr=Mr['text'].values; tbr=tfidf_acc(txr,yr,gr)
rev_rows=[]; rev_boot={}
for li,L in enumerate(dr['layers']):
    X=Xr[:,li,:].astype(np.float32)
    corr=oof_predictions(X,yr,gr); racc_c=oof_predictions(X,yr,gr,shuffle=True)
    acc,lo,hi,boots=group_bootstrap_accuracy(corr,gr); racc,_,_,_=group_bootstrap_accuracy(racc_c,gr); rev_boot[int(L)]=boots
    rev_rows.append({'model':MODEL_KEY,'task':'spurious_vs_antispurious','layer':int(L),'acc':round(acc,3),
                     'acc_lo':round(lo,3),'acc_hi':round(hi,3),'tfidf':round(tbr,3),'random':round(racc,3)})
RV=pd.DataFrame(rev_rows); RV.to_csv(tag('reversal.csv'),index=False)

# direction vs word-order overlay
ca=PM[PM.task=='causal_vs_anti'].sort_values('layer')
fig,ax=plt.subplots(figsize=(7.5,4.8))
ax.plot(ca.layer,ca.acc,'o-',color='#2563eb',lw=2,label='causal-vs-anti (direction)'); ax.fill_between(ca.layer,ca.acc_lo,ca.acc_hi,color='#2563eb',alpha=.15)
ax.plot(RV.layer,RV.acc,'o-',color='#dc2626',lw=2,label='fork reversal (word-order)'); ax.fill_between(RV.layer,RV.acc_lo,RV.acc_hi,color='#dc2626',alpha=.15)
ax.axhline(0.5,ls='--',color='grey',alpha=.6,label='chance'); ax.set_ylim(.3,1.02); ax.set_xlabel('layer'); ax.set_ylabel('accuracy [95% CI]')
ax.set_title(f'{MODEL_KEY} · Stage 5: direction vs word-order baseline'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(tag('stage5_direction_vs_order.png')); plt.show()

# gap test (peak vs peak, unpaired group-bootstrap)
ca_pL=int(ca.sort_values('acc').iloc[-1]['layer']); rv_pL=int(RV.sort_values('acc').iloc[-1]['layer'])
ca_b=boot_store[('causal_vs_anti',ca_pL)]; rv_b=rev_boot[rv_pL]
gap,lo,hi,p_le0=unpaired_gap_distribution(ca_b,rv_b)
print(f'causal-vs-anti peak {ca_b.mean():.3f} (L{ca_pL}) | fork-reversal peak {rv_b.mean():.3f} (L{rv_pL})')
print(f'GAP = {gap:+.3f}  95% CI [{lo:+.3f},{hi:+.3f}]  P(gap<=0)={p_le0:.4f}')
print('VERDICT:', 'SIGNIFICANT — direction exceeds the word-order floor (headline survives)' if lo>0 else 'NOT significant — requalify')
pd.DataFrame([{'model':MODEL_KEY,'causal_anti_peak':ca_b.mean(),'reversal_peak':rv_b.mean(),'gap':gap,'ci_lo':lo,'ci_hi':hi,'p_le0':p_le0,'significant':bool(lo>0)}]).to_csv(tag('gap_test.csv'),index=False)


## §12 · Stage 6 — Probe–output alignment (the dissociation)
Out-of-fold hidden margin (toward correct direction) vs prompt-conditioned output margin (from Stage 1).
**Strong hidden + output near 0 + low r = encoded-but-not-said.** Also stratified by intensity.


In [ ]:
# output margin per id, oriented to correct direction (reuse Stage 1 — no model reload)
def _om(rr):
    f,rv=np.log(max(rr['p_forward'],1e-9)),np.log(max(rr['p_reverse'],1e-9))
    return (f-rv) if rr['category']=='causal' else ((rv-f) if rr['category']=='anti-causal' else np.nan)
om={r['id']:_om(r) for _,r in base_df.iterrows()}

from sklearn.model_selection import GroupKFold
d=np.load(tag('hidden_main.npz'),allow_pickle=True); Xall=d['X']; layer_idx=list(d['layers'])
M=pd.read_csv(tag('meta_main.csv')); cm=np.isin(M['category'].values,['causal','anti-causal']); Mca=M[cm].reset_index(drop=True)
best_L=int(PM[PM.task=='causal_vs_anti'].sort_values('acc').iloc[-1]['layer']); li=layer_idx.index(best_L)
Xca=Xall[cm,li,:].astype(np.float32); yca=Mca['category'].values; gca=Mca['pair_id'].values
k=min(5,len(np.unique(gca))); gkf=GroupKFold(k); hidden=np.full(len(yca),np.nan)
for tr,te in gkf.split(Xca,yca,groups=gca):
    nc=max(2,min(128,len(tr)-1,Xca.shape[1]))
    clf=make_pipeline(StandardScaler(),PCA(n_components=nc,random_state=0),LogisticRegression(max_iter=1000)); clf.fit(Xca[tr],yca[tr])
    df_=clf.decision_function(Xca[te]); sign=1.0 if clf.classes_[1]=='causal' else -1.0; hidden[te]=df_*sign
Mca['hidden_margin_correct']=np.where(Mca['category'].values=='causal',hidden,-hidden)
Mca['output_margin']=Mca['id'].map(om).values
al=Mca.dropna(subset=['hidden_margin_correct','output_margin'])
r,lo,hi=pearson_boot(al.hidden_margin_correct.values,al.output_margin.values); rho,_=stats.spearmanr(al.hidden_margin_correct,al.output_margin)
print(f'Pearson r={r:.3f} [{lo:.3f},{hi:.3f}] | Spearman rho={rho:.3f} | n={len(al)} | best layer {best_L}')
Mca.to_csv(tag('alignment.csv'),index=False)
pd.DataFrame([{'model':MODEL_KEY,'pearson_r':r,'r_lo':lo,'r_hi':hi,'spearman_rho':rho,'n':len(al),'best_layer':best_L}]).to_csv(tag('alignment_stats.csv'),index=False)

fig,ax=plt.subplots(figsize=(6.3,6))
sns.scatterplot(data=al,x='hidden_margin_correct',y='output_margin',hue='category',
                palette={'causal':'#2563eb','anti-causal':'#f59e0b'},s=55,alpha=.8,ax=ax)
sns.regplot(data=al,x='hidden_margin_correct',y='output_margin',scatter=False,color='#475569',ci=95,ax=ax)
ax.axhline(0,ls='--',color='grey',alpha=.5); ax.axvline(0,ls='--',color='grey',alpha=.5)
ax.set_xlabel('hidden margin (correct direction, out-of-fold)'); ax.set_ylabel('output margin (logP correct − wrong)')
ax.set_title(f'{MODEL_KEY} · Stage 6: probe–output alignment\nr={r:.2f} [{lo:.2f},{hi:.2f}], ρ={rho:.2f}')
plt.tight_layout(); plt.savefig(tag('stage6_alignment.png')); plt.show()
print('strong hidden + output≈0 + low r ⇒ direction encoded internally, not read out.')


## §13 · Stage 7 — Cross-model accumulator + scaling (run after each model)
Appends this model's headline numbers to one master CSV and regenerates scaling curves (after ≥2 models).


In [ ]:
MASTER=f'{OUT}/MASTER_{DATA_VERSION}_results.csv'
def _causal(dd,col):
    v=dd[dd.category=='causal'][col].dropna().values; return bootstrap_ci_mean(v)[0]
row={'model':MODEL_KEY,'size_b':MODEL_SIZE_B,'run_date':RUN_DATE}
row['causal_anti_peak']=PM[PM.task=='causal_vs_anti'].acc.max()
row['reversal_peak']=RV.acc.max()
gt=pd.read_csv(tag('gap_test.csv')); row['gap']=gt['gap'].iloc[0]; row['gap_significant']=bool(gt['significant'].iloc[0])
als=pd.read_csv(tag('alignment_stats.csv')); row['alignment_r']=als['pearson_r'].iloc[0]
# dose headlines
gp=gdf.pivot_table(index=['id','pair_id','category'],columns='level',values='p_correct').reset_index(); gp['dh']=gp['high']-gp['base']
row['dose_graded_causal_dhigh']=_causal(gp.assign(category=gp.category),'dh')
row['dose_moreless_causal_dmore']=bootstrap_ci_mean(piv[piv.category=='causal']['delta_more'].dropna().values)[0]
s3=pd.read_csv(tag('stage3B_corr.csv')); cc=s3[s3.category=='causal']; row['stage3B_causal_cooc_r']=cc['r'].iloc[0] if len(cc) else np.nan
existing=pd.read_csv(MASTER) if os.path.exists(MASTER) else pd.DataFrame(columns=list(row.keys()))
existing=existing[existing.model!=MODEL_KEY] if 'model' in existing.columns else existing
master=pd.concat([existing,pd.DataFrame([row])],ignore_index=True); master.to_csv(MASTER,index=False)
print(f'master updated: {len(master)} models'); print(master[['model','size_b','causal_anti_peak','reversal_peak','gap_significant','alignment_r','stage3B_causal_cooc_r']].to_string(index=False))


In [ ]:
# scaling curves (after >=2 models)
if os.path.exists(MASTER):
    m=pd.read_csv(MASTER).sort_values('size_b')
    if len(m)>=2:
        fig,axes=plt.subplots(1,3,figsize=(16,4.6))
        a=axes[0]; a.plot(m.size_b,m.causal_anti_peak,'o-',color='#2563eb',lw=2,label='causal-vs-anti'); a.plot(m.size_b,m.reversal_peak,'o-',color='#dc2626',lw=2,label='fork-reversal'); a.axhline(.5,ls='--',color='grey',alpha=.5)
        a.set_title('direction vs word-order, by scale'); a.set_ylabel('peak probe acc'); a.legend(fontsize=8)
        b=axes[1]; b.plot(m.size_b,m.alignment_r,'o-',color=GAP_COLOR,lw=2); b.axhline(0,ls='--',color='grey',alpha=.5); b.set_title('probe–output alignment r, by scale'); b.set_ylabel('Pearson r')
        c=axes[2]; c.plot(m.size_b,m.stage3B_causal_cooc_r,'o-',color='#16a34a',lw=2); c.axhline(0,ls='--',color='grey',alpha=.5); c.set_title('Stage 3B co-occurrence r (causal), by scale'); c.set_ylabel('corr(Δcooc,Δp)')
        for ax in axes:
            ax.set_xscale('log'); xt=sorted(m.size_b.tolist()); ax.set_xticks(xt); ax.set_xticklabels([f'{x:.0f}' for x in xt],rotation=45,fontsize=7); ax.set_xlabel('params (B)')
        plt.suptitle(f'Cross-model scaling (n={len(m)} base models) · {RUN_DATE}',y=1.03); plt.tight_layout()
        plt.savefig(f'{OUT}/MASTER_{DATA_VERSION}_{RUN_DATE}_scaling.png'); plt.show()
    else: print('need >=2 models for scaling; have:',list(m.model))


## §14 · Stage 8 — Dose geometry (do intensity-induced hidden shifts carry causal signal?)
Per-relation hidden shift Δ = h(high) − h(base) at each layer. Learn a causal−spurious shift direction on a
**GroupKFold train split**, project held-out shifts: if causal projects higher than spurious, the dose shift is
structurally causal, not generic. Held-out, no circularity.


In [ ]:
# extract base+high hidden states for causal+spurious (geometry needs levels)
geo_items=[it for it in items if it['category'] in ('causal','spurious')]
gX=[]; gids=[]; glvls=[]; gcat=[]
for it in geo_items:
    for lvl in ['base','high']:
        gX.append(hidden_for(hs_prompt(x_for(it,lvl), it['y_text'])).astype(np.float16))
        gids.append(it['id']); glvls.append(lvl); gcat.append(it['category'])
gX=np.stack(gX); gids=np.array(gids); glvls=np.array(glvls); gcat=np.array(gcat); glayers=selected_layers
idx={(gids[r],glvls[r]):r for r in range(len(gids))}

geo=[]
for li,L in enumerate(glayers):
    rows_d=[]
    for rid in np.unique(gids):
        if (rid,'base') not in idx or (rid,'high') not in idx: continue
        hb=gX[idx[(rid,'base')],li,:].astype(np.float32); hh=gX[idx[(rid,'high')],li,:].astype(np.float32)
        cat=gcat[gids==rid][0]; rows_d.append({'id':rid,'category':cat,'delta':hh-hb})
    dd=pd.DataFrame(rows_d)
    sub=dd[dd.category.isin(['causal','spurious'])].reset_index(drop=True)
    if sub.category.nunique()<2: continue
    gkf=GroupKFold(n_splits=min(5,sub.id.nunique())); pc,ps=[],[]
    for tr,te in gkf.split(sub,sub.category,groups=sub.id):
        td=sub.iloc[tr]
        dC=np.mean(np.stack(td[td.category=='causal']['delta'].values),0); dS=np.mean(np.stack(td[td.category=='spurious']['delta'].values),0)
        direction=dC-dS; direction/=(np.linalg.norm(direction)+1e-8)
        for _,rr in sub.iloc[te].iterrows():
            (pc if rr.category=='causal' else ps).append(float(np.dot(rr['delta'],direction)))
    if len(pc)>2 and len(ps)>2:
        u,p=stats.mannwhitneyu(pc,ps,alternative='greater')
        geo.append({'layer':int(L),'causal_proj':round(np.mean(pc),3),'spurious_proj':round(np.mean(ps),3),'p':round(p,4),'auc':round(u/(len(pc)*len(ps)),3)})
        print(f'  L{L}: causal proj {np.mean(pc):+.3f} vs spurious {np.mean(ps):+.3f}  p={p:.4f} AUC={u/(len(pc)*len(ps)):.3f}')
GEO=pd.DataFrame(geo); GEO.to_csv(tag('stage8_dose_geometry.csv'),index=False)
if len(GEO):
    fig,ax=plt.subplots(figsize=(7,4.4))
    ax.plot(GEO.layer,GEO.auc,'o-',color=GAP_COLOR,lw=2,ms=7); ax.axhline(.5,ls='--',color='grey',alpha=.6,label='chance')
    ax.set_xlabel('layer'); ax.set_ylabel('held-out AUC (causal>spurious shift)'); ax.set_ylim(.3,1.02)
    ax.set_title(f'{MODEL_KEY} · Stage 8: dose-shift geometry'); ax.legend()
    plt.tight_layout(); plt.savefig(tag('stage8_dose_geometry.png')); plt.show()


## §15 · Stage 9 — Interventions (causal test, last)
Escalation ladder: **9.1 mean-vector steering** → **9.2 cause-token patching**. Both reuse the loaded model.
Build a dose vector / transplant a high-dose activation and ask if base prompts behave more like high-dose,
*more for causal than fork-spurious*. Continuation held to base form throughout; α=+1 is the principled
recreate-high magnitude; controls = sign-flip, matched-norm random, category.


In [ ]:
# 9.1 · mean-vector dose steering — build v_dose, sweep alpha, controls
def get_decoder_layers(m):
    for path in ['model.layers','gpt_neox.layers','transformer.h','model.decoder.layers']:
        obj=m; ok=True
        for a in path.split('.'):
            if hasattr(obj,a): obj=getattr(obj,a)
            else: ok=False; break
        if ok: return obj
    raise ValueError('no decoder layers')
LAYERS=get_decoder_layers(model); RNG_S=np.random.default_rng(0)
MAIN_LAYER=n_layers//2; ALPHAS=[-3,-2,-1,0,1,2,3]; SPAN='from_last_prompt'
causal_all=[it for it in items if it['category']=='causal']; anti_all=[it for it in items if it['category']=='anti-causal']; spur_all=[it for it in items if it['category']=='spurious']
cp=sorted({it['pair_id'] for it in causal_all}); RNG_S.shuffle(cp); h=len(cp)//2
tr_p,te_p=set(cp[:h]),set(cp[h:])
c_build=[it for it in causal_all if it['pair_id'] in tr_p][:150]; c_eval=[it for it in causal_all if it['pair_id'] in te_p][:100]
a_eval=anti_all[:100]; s_eval=spur_all[:100]

class Steer:
    def __init__(s,L): s.L=L; s.h=None; s.cur=None; s.v=None; s.st=0; s.en=None
    def _hk(s,m,i,o):
        if s.v is None: return o
        H=o[0] if isinstance(o,tuple) else o; H[:,s.st:s.en,:]=H[:,s.st:s.en,:]+s.v.to(H.device,H.dtype)
        return (H,)+tuple(o[1:]) if isinstance(o,tuple) else H
    def tgt(s,li):
        if s.cur==li and s.h is not None: return
        if s.h is not None: s.h.remove()
        s.h=s.L[li].register_forward_hook(s._hk); s.cur=li
    def set(s,v,st,en): s.v,s.st,s.en=v,st,en
    def clr(s): s.v=None
    def rm(s):
        if s.h is not None: s.h.remove()
        s.h=None; s.cur=None
ST=Steer(LAYERS)
@torch.no_grad()
def last_hidden(prompt,li):
    store={}
    def cap(m,i,o):
        H=o[0] if isinstance(o,tuple) else o; store['h']=H[0,-1,:].detach().float().cpu()
    hd=LAYERS[li].register_forward_hook(cap); model(tok(prompt,return_tensors='pt').input_ids.to(model.device)); hd.remove(); return store['h']
@torch.no_grad()
def clp_s(prompt,cont,v=None,li=None,span='from_last_prompt'):
    p=tok(prompt,return_tensors='pt').input_ids.to(model.device); full=tok(prompt+cont,return_tensors='pt').input_ids.to(model.device); n=p.shape[1]
    if full.shape[1]<=n: return float('-inf')
    if v is not None: ST.tgt(li); ST.set(v,n-1,None if span=='from_last_prompt' else n)
    out=model(full).logits
    if v is not None: ST.clr()
    lp=F.log_softmax(out,dim=-1); return lp[0,n-1:-1,:].gather(-1,full[0,n:].unsqueeze(-1)).squeeze(-1).mean().item()
def margin_s(prompt,xb,yb,v=None,li=None,span='from_last_prompt'):
    return clp_s(prompt,f'{xb} directly causes {yb}',v,li,span)-clp_s(prompt,f'{yb} directly causes {xb}',v,li,span)
def build_vec(lst,li):
    return torch.stack([last_hidden(NEUTRAL_PROMPT.format(x=x_for(it,'high'),y=it['y_text']),li)-last_hidden(NEUTRAL_PROMPT.format(x=it['x_text'],y=it['y_text']),li) for it in lst]).mean(0)

v_dose=build_vec(c_build,MAIN_LAYER); vn=v_dose.norm().item()
rand_vecs=[(lambda r:r/r.norm()*vn)(torch.randn(v_dose.shape[0],generator=torch.Generator().manual_seed(100+s))) for s in range(3)]
print(f'||v_dose||={vn:.3f}')
rows=[]
for cat,lst in [('causal',c_eval),('anti-causal',a_eval),('fork_spurious',s_eval)]:
    for it in lst:
        xb,yb=it['x_text'],it['y_text']; pb=NEUTRAL_PROMPT.format(x=xb,y=yb); mb=margin_s(pb,xb,yb)
        for a in ALPHAS:
            ms=mb if a==0 else margin_s(pb,xb,yb,a*v_dose,MAIN_LAYER,SPAN)
            rows.append({'category':cat,'pair_id':it['pair_id'],'alpha':a,'delta':ms-mb})
sweep=pd.DataFrame(rows); sweep.to_csv(tag('stage9_steering_sweep.csv'),index=False)
fig,ax=plt.subplots(figsize=(7,4.6))
for cat in ['causal','anti-causal','fork_spurious']:
    dd=sweep[sweep.category==cat]; ms=[];los=[];his=[]
    for a in ALPHAS:
        m,lo,hi=grouped_bootstrap_ci(dd[dd.alpha==a],'delta','pair_id'); ms+=[m];los+=[lo];his+=[hi]
    ax.plot(ALPHAS,ms,'o-',color=CAT_COLOR_S[cat],lw=2.2,ms=6,label=cat); ax.fill_between(ALPHAS,los,his,color=CAT_COLOR_S[cat],alpha=.15)
ax.axvline(1,ls=':',color='#dc2626',alpha=.8,label='α=+1'); ax.axhline(0,ls='--',color='grey',alpha=.5)
ax.set_xlabel('alpha (× v_dose)'); ax.set_ylabel('Δ margin (steered−base) [95% CI]'); ax.set_title(f'{MODEL_KEY} · Stage 9.1: dose steering'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(tag('stage9_steering.png')); plt.show()
a1=sweep[sweep.alpha==1]
for cat in ['causal','anti-causal','fork_spurious']:
    m,lo,hi=grouped_bootstrap_ci(a1[a1.category==cat],'delta','pair_id'); print(f'  α=+1 {cat:14s} Δ={m:+.4f} [{lo:+.4f},{hi:+.4f}] {"★" if lo>0 else ""}')
ST.rm()


In [ ]:
# 9.2 · cause-token / X-token activation patching (localized dose test)
PREFIX=NEUTRAL_PROMPT.split('{x}')[0]; XSTART=len(PREFIX); CAUSE_PATCH_N=30
CAUSE_LAYERS=[L for L in [MAIN_LAYER, n_layers*3//4, n_layers-2] if 0<=L<n_layers]
class SpanPatch:
    def __init__(s,L): s.L=L; s.h=None; s.cur=None; s.v=None; s.idx=None
    def _hk(s,m,i,o):
        if s.v is None: return o
        H=o[0] if isinstance(o,tuple) else o; H[0,s.idx,:]=s.v.to(H.device,H.dtype)
        return (H,)+tuple(o[1:]) if isinstance(o,tuple) else H
    def tgt(s,li):
        if s.cur==li and s.h is not None: return
        if s.h is not None: s.h.remove()
        s.h=s.L[li].register_forward_hook(s._hk); s.cur=li
    def set(s,v,idx): s.v,s.idx=v,idx
    def clr(s): s.v=None
    def rm(s):
        if s.h is not None: s.h.remove()
        s.h=None; s.cur=None
SP=SpanPatch(LAYERS)
def find_span(prompt,cs,ce):
    if not getattr(tok,'is_fast',False): return None
    try: offs=tok(prompt,return_offsets_mapping=True)['offset_mapping']
    except Exception: return None
    sp=[i for i,(a,b) in enumerate(offs) if (b>a) and (a<ce) and (b>cs)]; return sp or None
@torch.no_grad()
def cap_span(prompt,span,li):
    store={}
    def cap(m,i,o):
        H=o[0] if isinstance(o,tuple) else o; store['h']=H[0,span,:].detach().clone()
    hd=LAYERS[li].register_forward_hook(cap); model(tok(prompt,return_tensors='pt').input_ids.to(model.device)); hd.remove(); return store['h']
@torch.no_grad()
def clp_p(prompt,cont,v=None,idx=None,li=None):
    p=tok(prompt,return_tensors='pt').input_ids.to(model.device); full=tok(prompt+cont,return_tensors='pt').input_ids.to(model.device); n=p.shape[1]
    if full.shape[1]<=n: return float('-inf')
    if v is not None: SP.tgt(li); SP.set(v,idx)
    out=model(full).logits
    if v is not None: SP.clr()
    lp=F.log_softmax(out,dim=-1); return lp[0,n-1:-1,:].gather(-1,full[0,n:].unsqueeze(-1)).squeeze(-1).mean().item()
def margin_p(prompt,xb,yb,v=None,idx=None,li=None):
    return clp_p(prompt,f'{xb} directly causes {yb}',v,idx,li)-clp_p(prompt,f'{yb} directly causes {xb}',v,idx,li)

prep={c:[] for c in ['causal','anti-causal','spurious']}; skipped=0
for c in prep:
    for it in [x for x in items if x['category']==c][:CAUSE_PATCH_N]:
        xb,yb=it['x_text'],it['y_text']; hi=x_for(it,'high')
        bsp=find_span(NEUTRAL_PROMPT.format(x=xb,y=yb),XSTART,XSTART+len(xb))
        pos=hi.rfind(xb); hsp=find_span(NEUTRAL_PROMPT.format(x=hi,y=yb),XSTART+pos,XSTART+pos+len(xb)) if pos>=0 else None
        if not bsp or not hsp: skipped+=1; continue
        prep[c].append(dict(it=it,xb=xb,yb=yb,pb=NEUTRAL_PROMPT.format(x=xb,y=yb),ph=NEUTRAL_PROMPT.format(x=hi,y=yb),bsp=bsp,hsp=hsp,pid=it['pair_id']))
print('cause-token prepared:',{c:len(prep[c]) for c in prep},'skipped:',skipped)
prows=[]
for L in CAUSE_LAYERS:
    for c in prep:
        for r in prep[c]: r['ha']=cap_span(r['ph'],r['hsp'],L)
    for c in prep:
        pool=prep[c]
        for j,r in enumerate(pool):
            ml=min(len(r['bsp']),len(r['hsp'])); idx=r['bsp'][:ml]
            mb=margin_p(r['pb'],r['xb'],r['yb']); mh=margin_p(r['ph'],r['xb'],r['yb'])
            mp=margin_p(r['pb'],r['xb'],r['yb'],r['ha'][:ml],idx,L)
            others=[k for k in range(len(pool)) if k!=j]; rr=pool[int(RNG_S.choice(others))] if others else r
            mlr=min(len(r['bsp']),len(rr['hsp'])); mpr=margin_p(r['pb'],r['xb'],r['yb'],rr['ha'][:mlr],r['bsp'][:mlr],L)
            prows.append({'category':c,'pair_id':r['pid'],'layer':L,'d_patch':mp-mb,'patch_minus_random':mp-mpr,'recovered':(mp-mb)/(mh-mb) if abs(mh-mb)>1e-4 else np.nan})
    print(f'  layer {L} done')
P=pd.DataFrame(prows); P.to_csv(tag('stage9_causetoken_patch.csv'),index=False)
fig,axes=plt.subplots(1,2,figsize=(13,4.6))
for ax,col,ttl in [(axes[0],'d_patch','cause-token patch effect'),(axes[1],'patch_minus_random','patch − random')]:
    for c in ['causal','anti-causal','spurious']:
        ms=[];los=[];his=[]
        for L in CAUSE_LAYERS:
            d=P[(P.category==c)&(P.layer==L)].dropna(subset=[col])
            if d['pair_id'].nunique()<2: ms+=[np.nan];los+=[np.nan];his+=[np.nan]; continue
            m,lo,hi=grouped_bootstrap_ci(d,col,'pair_id'); ms+=[m];los+=[lo];his+=[hi]
        ax.plot(CAUSE_LAYERS,ms,'o-',color=PALETTE[c],lw=2.2,ms=6,label=c); ax.fill_between(CAUSE_LAYERS,los,his,color=PALETTE[c],alpha=.15)
    ax.axhline(0,ls='--',color='grey',alpha=.5); ax.set_xlabel('patch layer'); ax.set_title(ttl); ax.legend(fontsize=8)
plt.suptitle(f'{MODEL_KEY} · Stage 9.2: cause-token patching',y=1.02); plt.tight_layout(); plt.savefig(tag('stage9_causetoken.png')); plt.show()
SP.rm()
print('causal patch−random > 0 with fork ≈ 0 ⇒ dose is localized & causal at the cause token.')


In [ ]:
# ============================================================
# Recover behavioural dataframe as `bdf`
# Run this before Stage 10 if bdf is not defined.
# ============================================================

import pandas as pd
import numpy as np

def looks_like_bdf(obj):
    if not isinstance(obj, pd.DataFrame):
        return False
    needed = {"id", "category", "level", "p_correct"}
    return needed.issubset(set(obj.columns))

candidates = []
for name, obj in list(globals().items()):
    if name.startswith("_"):
        continue
    if looks_like_bdf(obj):
        candidates.append((name, len(obj), list(obj.columns)))

print("Candidate behavioural dataframes:")
for name, n, cols in candidates:
    print(f"  {name:30s} n={n:6d} cols={cols[:12]}")

if not candidates:
    raise ValueError(
        "No behavioural dataframe found with columns id, category, level, p_correct.\n"
        "Run the scoring/Stage 2 cells first, or tell me the dataframe name printed in your notebook."
    )

# Prefer largest candidate
best_name = sorted(candidates, key=lambda x: x[1], reverse=True)[0][0]
bdf = globals()[best_name].copy()

print(f"\nUsing `{best_name}` as bdf")
print("bdf shape:", bdf.shape)
print("levels:", sorted(bdf["level"].astype(str).unique()))
print("categories:", bdf["category"].value_counts().to_dict())
display(bdf.head())

In [ ]:
# ============================================================
# Stage 10 — Local dose-shift manifold / heterogeneity diagnostic
# ============================================================
#
# Motivation:
#   Stage 8: hidden dose-shift geometry separates causal/spurious.
#   Stage 9.1: global mean-vector steering is imperfect/model-dependent.
#   Stage 9.2: cause-token patching fails.
#
# Hypothesis:
#   The dose signal is distributed and relation-conditioned:
#   not one global knob, not localized at the cause token.
#
# Tests:
#   1. How much of delta_h is explained by one global mean vector?
#   2. Do dose-shift vectors cluster by relation type?
#   3. Does local kNN structure predict category better than global projection?
#   4. Does local hidden geometry predict behavioural dose effect better
#      than a single global projection?
# ============================================================

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from scipy import stats
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import accuracy_score, roc_auc_score, r2_score
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

# ----------------------------
# Config
# ----------------------------

STAGE10_RANDOM_SEED = 42
STAGE10_MAX_ITEMS_PER_CATEGORY = 300   # use 80 for quick test, 300 for final

# Pick best layer manually if you know it.
# Pythia: 18 was good.
# Qwen-14B: 24 looked good.
if "qwen" in MODEL_KEY.lower():
    STAGE10_LAYER = 24
elif "pythia" in MODEL_KEY.lower():
    STAGE10_LAYER = 18
else:
    STAGE10_LAYER = None  # auto below

K_LIST = [3, 5, 10, 20]

# Prefer old graded dose if available.
levels_available = set(bdf["level"].astype(str).unique())
if {"high", "base"}.issubset(levels_available):
    POS_LEVEL = "high"
    NEG_LEVEL = "base"
    CONTRAST_NAME = "high_minus_base"
elif {"more", "less"}.issubset(levels_available):
    POS_LEVEL = "more"
    NEG_LEVEL = "less"
    CONTRAST_NAME = "more_minus_less"
else:
    raise ValueError(f"Need either high/base or more/less levels. Found: {levels_available}")

print(f"MODEL_KEY = {MODEL_KEY}")
print(f"Stage 10 contrast = {POS_LEVEL} - {NEG_LEVEL}")

# ----------------------------
# Helpers: recover text columns
# ----------------------------

def find_xy_cols(df):
    x_cands = ["x_text", "X", "x", "cause", "cause_text", "var_x", "source", "treatment"]
    y_cands = ["y_text", "Y", "y", "effect", "effect_text", "var_y", "target", "outcome"]
    cols_lower = {c.lower(): c for c in df.columns}
    x_col, y_col = None, None
    for c in x_cands:
        if c.lower() in cols_lower:
            x_col = cols_lower[c.lower()]
            break
    for c in y_cands:
        if c.lower() in cols_lower:
            y_col = cols_lower[c.lower()]
            break
    return x_col, y_col

def as_df_maybe(obj):
    if isinstance(obj, pd.DataFrame):
        return obj.copy()
    if isinstance(obj, list) and len(obj) and isinstance(obj[0], dict):
        return pd.DataFrame(obj)
    if isinstance(obj, dict):
        try:
            df = pd.DataFrame(obj)
            if len(df):
                return df
        except Exception:
            pass
        try:
            df = pd.DataFrame(list(obj.values()))
            if len(df):
                return df
        except Exception:
            pass
    return None

bdf_s10 = bdf.copy()
bdf_s10["id"] = bdf_s10["id"].astype(str)

if "x_text" not in bdf_s10.columns or "y_text" not in bdf_s10.columns:
    candidate_sources = []
    for name, obj in list(globals().items()):
        if name.startswith("_"):
            continue
        df_try = as_df_maybe(obj)
        if df_try is None or "id" not in df_try.columns:
            continue
        x_col, y_col = find_xy_cols(df_try)
        if x_col is not None and y_col is not None:
            overlap = len(set(df_try["id"].astype(str)) & set(bdf_s10["id"].astype(str)))
            candidate_sources.append((name, df_try, x_col, y_col, overlap, len(df_try)))

    if not candidate_sources:
        raise ValueError("Could not find any source with id + x/y text columns.")

    candidate_sources = sorted(candidate_sources, key=lambda t: t[4], reverse=True)
    src_name, src_df, x_col, y_col, overlap, n_src = candidate_sources[0]
    print(f"Using text source: {src_name} | overlap={overlap} | x={x_col} y={y_col}")

    item_text = src_df[["id", x_col, y_col]].drop_duplicates("id").copy()
    item_text["id"] = item_text["id"].astype(str)
    item_text = item_text.rename(columns={x_col: "x_text", y_col: "y_text"})
    item_text["x_text"] = item_text["x_text"].astype(str)
    item_text["y_text"] = item_text["y_text"].astype(str)

    bdf_s10 = bdf_s10.merge(item_text, on="id", how="left")

if bdf_s10["x_text"].isna().any():
    raise ValueError("Some rows are still missing x_text/y_text after merge.")

# ----------------------------
# Prompt builder
# Keep this aligned with your notebook.
# ----------------------------

def level_x(x, level):
    x = str(x)
    if level == "base":
        return x
    if level == "low":
        return "weak " + x
    if level == "moderate":
        return "moderate " + x
    if level == "high":
        return "strong " + x
    if level == "more":
        return "more " + x
    if level == "less":
        return "less " + x
    return x

def make_prompt(x, y, level):
    lx = level_x(x, level)
    return f"Consider the variables: {lx} and {y}. The relation between them is that "

# ----------------------------
# Get layers
# ----------------------------

def get_layers(model):
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    if hasattr(model, "gpt_neox") and hasattr(model.gpt_neox, "layers"):
        return model.gpt_neox.layers
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        return model.transformer.h
    raise ValueError("Could not find transformer layers.")

layers = get_layers(model)
n_layers = len(layers)

if STAGE10_LAYER is None:
    STAGE10_LAYER = int(round(0.5 * (n_layers - 1)))

STAGE10_LAYER = min(max(0, STAGE10_LAYER), n_layers - 1)
print(f"Using layer {STAGE10_LAYER} / {n_layers-1}")

@torch.no_grad()
def get_prompt_hidden(prompt, layer_idx):
    cache = {}

    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        cache["h"] = h[:, -1, :].detach().float().cpu()
        return out

    handle = layers[layer_idx].register_forward_hook(hook)
    ids = tok(prompt, return_tensors="pt").input_ids.to(model.device)
    _ = model(ids)
    handle.remove()
    return cache["h"].squeeze(0).numpy()

# ----------------------------
# Prepare item sample
# ----------------------------

cols = ["id", "category", "x_text", "y_text"]
if "polarity" in bdf_s10.columns:
    cols.append("polarity")
if "pair_id" in bdf_s10.columns:
    cols.append("pair_id")

item_df = bdf_s10.drop_duplicates("id")[cols].copy()

sampled = []
for cat in ["causal", "anti-causal", "spurious"]:
    sub = item_df[item_df["category"] == cat].copy()
    if len(sub) > STAGE10_MAX_ITEMS_PER_CATEGORY:
        sub = sub.sample(STAGE10_MAX_ITEMS_PER_CATEGORY, random_state=STAGE10_RANDOM_SEED)
    sampled.append(sub)

item_sample = pd.concat(sampled, ignore_index=True)
print("\nStage 10 sample:")
print(item_sample["category"].value_counts())

# ----------------------------
# Compute hidden dose shifts: delta_h = h(pos) - h(neg)
# ----------------------------

rows = []
deltas = []

for _, row in tqdm(item_sample.iterrows(), total=len(item_sample), desc="Computing hidden dose shifts"):
    x = row["x_text"]
    y = row["y_text"]

    p_pos = make_prompt(x, y, POS_LEVEL)
    p_neg = make_prompt(x, y, NEG_LEVEL)

    h_pos = get_prompt_hidden(p_pos, STAGE10_LAYER)
    h_neg = get_prompt_hidden(p_neg, STAGE10_LAYER)

    delta = h_pos - h_neg
    deltas.append(delta)

    rows.append({
        "id": row["id"],
        "pair_id": row["pair_id"] if "pair_id" in row else row["id"],
        "category": row["category"],
        "polarity": row["polarity"] if "polarity" in row else "unknown",
    })

D = np.stack(deltas).astype(np.float32)
meta = pd.DataFrame(rows)

print("D shape:", D.shape)

# ----------------------------
# Behavioural dose effect per item
# ----------------------------

piv = bdf_s10.pivot_table(
    index=["id", "category"],
    columns="level",
    values="p_correct",
    aggfunc="mean"
).reset_index()
piv.columns = [str(c) for c in piv.columns]

if POS_LEVEL in piv.columns and NEG_LEVEL in piv.columns:
    piv["behav_effect"] = piv[POS_LEVEL] - piv[NEG_LEVEL]
else:
    piv["behav_effect"] = np.nan

meta = meta.merge(piv[["id", "behav_effect"]], on="id", how="left")

# ----------------------------
# Basic vector helpers
# ----------------------------

def unit(v, eps=1e-8):
    return v / (np.linalg.norm(v) + eps)

def cosine(a, b):
    return float(np.dot(unit(a), unit(b)))

def bootstrap_mean(x, n_boot=3000, seed=42):
    x = np.asarray(pd.Series(x).dropna().values, dtype=float)
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = rng.choice(x, size=(n_boot, len(x)), replace=True).mean(axis=1)
    return float(x.mean()), float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

# ----------------------------
# Test 1: global vector concentration
# ----------------------------

global_mean = D.mean(axis=0)
global_u = unit(global_mean)

proj_scalar = D @ global_u
proj_vec = np.outer(proj_scalar, global_u)
frac_global_energy = np.sum(proj_vec ** 2, axis=1) / (np.sum(D ** 2, axis=1) + 1e-8)
cos_global = np.array([cosine(d, global_mean) for d in D])

meta["proj_global"] = proj_scalar
meta["frac_global_energy"] = frac_global_energy
meta["cos_to_global"] = cos_global

print("\n" + "="*80)
print("TEST 1 — How global is the dose vector?")
print("="*80)

for cat in ["causal", "anti-causal", "spurious"]:
    sub = meta[meta["category"] == cat]
    m, lo, hi = bootstrap_mean(100 * sub["frac_global_energy"])
    c, clo, chi = bootstrap_mean(sub["cos_to_global"])
    p, plo, phi = bootstrap_mean(sub["proj_global"])
    print(f"{cat:12s} global-energy%={m:6.2f} [{lo:6.2f},{hi:6.2f}] | cos={c:+.3f} [{clo:+.3f},{chi:+.3f}] | proj={p:+.3f} [{plo:+.3f},{phi:+.3f}]")

# Category mean-vector cosines
cat_means = {}
for cat in ["causal", "anti-causal", "spurious"]:
    cat_means[cat] = D[meta["category"].eq(cat).values].mean(axis=0)

print("\nCategory mean-vector cosines:")
for a in ["causal", "anti-causal", "spurious"]:
    print(
        f"{a:12s} "
        f"cos to causal={cosine(cat_means[a], cat_means['causal']):+.3f} | "
        f"cos to anti={cosine(cat_means[a], cat_means['anti-causal']):+.3f} | "
        f"cos to spur={cosine(cat_means[a], cat_means['spurious']):+.3f}"
    )

# ----------------------------
# Test 2: local neighbour consistency
# ----------------------------

print("\n" + "="*80)
print("TEST 2 — Do dose-shift vectors have same-category neighbours?")
print("="*80)

# Cosine similarity on normalized deltas
D_norm = D / (np.linalg.norm(D, axis=1, keepdims=True) + 1e-8)
S = D_norm @ D_norm.T
np.fill_diagonal(S, -np.inf)

cats = meta["category"].values
local_rows = []

for k in K_LIST:
    same_fracs = []
    causal_neighbor_fracs = []
    for i in range(len(meta)):
        nn = np.argsort(-S[i])[:k]
        same = np.mean(cats[nn] == cats[i])
        same_fracs.append(same)

        if cats[i] == "causal":
            causal_neighbor_fracs.append(np.mean(cats[nn] == "causal"))

    meta[f"knn_samecat_k{k}"] = same_fracs

    overall = float(np.mean(same_fracs))
    chance = sum((pd.Series(cats).value_counts(normalize=True) ** 2).values)
    causal_mean = float(np.mean(causal_neighbor_fracs)) if len(causal_neighbor_fracs) else np.nan

    local_rows.append({
        "k": k,
        "overall_same_category_neighbor_fraction": overall,
        "chance_same_category_fraction": chance,
        "causal_items_causal_neighbor_fraction": causal_mean,
    })

    print(f"k={k:2d}: same-category neighbours={overall:.3f} | chance≈{chance:.3f} | causal→causal neighbours={causal_mean:.3f}")

local_df = pd.DataFrame(local_rows)

# ----------------------------
# Test 3: global projection vs local kNN category prediction
# ----------------------------

print("\n" + "="*80)
print("TEST 3 — Category prediction: global projection vs local/full delta_h")
print("="*80)

label_map = {"causal": 0, "anti-causal": 1, "spurious": 2}
y3 = meta["category"].map(label_map).values

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=STAGE10_RANDOM_SEED)

# A. single global scalar projection
X_global = meta[["proj_global"]].values

global_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

global_acc = cross_val_score(global_clf, X_global, y3, cv=cv, scoring="accuracy").mean()

# B. linear full delta_h
linear_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, C=1.0, multi_class="auto")
)
linear_acc = cross_val_score(linear_clf, D, y3, cv=cv, scoring="accuracy").mean()

# C. kNN on full delta_h
knn_accs = {}
for k in K_LIST:
    knn = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    )
    knn_accs[k] = cross_val_score(knn, D, y3, cv=cv, scoring="accuracy").mean()

print(f"3-way category accuracy from GLOBAL PROJECTION only: {global_acc:.3f}")
print(f"3-way category accuracy from LINEAR full delta_h:     {linear_acc:.3f}")
for k, acc in knn_accs.items():
    print(f"3-way category accuracy from kNN full delta_h k={k:2d}: {acc:.3f}")

# Directed vs spurious version
yd = meta["category"].isin(["causal", "anti-causal"]).astype(int).values
global_acc_d = cross_val_score(global_clf, X_global, yd, cv=cv, scoring="accuracy").mean()
linear_acc_d = cross_val_score(linear_clf, D, yd, cv=cv, scoring="accuracy").mean()

knn_accs_d = {}
for k in K_LIST:
    knn = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k, metric="euclidean")
    )
    knn_accs_d[k] = cross_val_score(knn, D, yd, cv=cv, scoring="accuracy").mean()

print("\nDirected(causal/anti) vs spurious:")
print(f"binary acc from GLOBAL PROJECTION only: {global_acc_d:.3f}")
print(f"binary acc from LINEAR full delta_h:     {linear_acc_d:.3f}")
for k, acc in knn_accs_d.items():
    print(f"binary acc from kNN full delta_h k={k:2d}: {acc:.3f}")

# ----------------------------
# Test 4: behavioural dose prediction
# ----------------------------

print("\n" + "="*80)
print("TEST 4 — Can hidden dose geometry predict behavioural dose effect?")
print("="*80)

ok = meta["behav_effect"].notna().values
Xg = X_global[ok]
Xd = D[ok]
yb = meta.loc[ok, "behav_effect"].values

if len(yb) >= 50:
    kf = KFold(n_splits=5, shuffle=True, random_state=STAGE10_RANDOM_SEED)

    ridge_global = make_pipeline(
        StandardScaler(),
        RidgeCV(alphas=np.logspace(-4, 4, 25))
    )
    r2_global = cross_val_score(ridge_global, Xg, yb, cv=kf, scoring="r2")
    r2_linear = cross_val_score(ridge_global, Xd, yb, cv=kf, scoring="r2")

    print(f"CV R² from GLOBAL projection only: mean={r2_global.mean():+.3f} scores={np.round(r2_global,3)}")
    print(f"CV R² from LINEAR full delta_h:     mean={r2_linear.mean():+.3f} scores={np.round(r2_linear,3)}")

    for k in K_LIST:
        knnr = make_pipeline(
            StandardScaler(),
            KNeighborsRegressor(n_neighbors=k, metric="euclidean")
        )
        r2_knn = cross_val_score(knnr, Xd, yb, cv=kf, scoring="r2")
        print(f"CV R² from LOCAL kNN full delta_h k={k:2d}: mean={r2_knn.mean():+.3f} scores={np.round(r2_knn,3)}")

    print("\nInterpretation:")
    print("- global R² low but category prediction high => hidden geometry is categorical, not direct output readout.")
    print("- kNN/local R² > global R² => local manifold predicts behaviour better than one vector.")
else:
    print("Not enough behavioural effect rows for regression.")

# ----------------------------
# Statistical comparison: causal vs spurious global projection
# ----------------------------

print("\n" + "="*80)
print("Causal vs spurious projection test")
print("="*80)

ca = meta[meta["category"] == "causal"]["proj_global"].values
sp = meta[meta["category"] == "spurious"]["proj_global"].values

u, p = stats.mannwhitneyu(ca, sp, alternative="greater")
auc = u / (len(ca) * len(sp))
m_ca, lo_ca, hi_ca = bootstrap_mean(ca)
m_sp, lo_sp, hi_sp = bootstrap_mean(sp)

print(f"causal proj   {m_ca:+.3f} [{lo_ca:+.3f},{hi_ca:+.3f}] n={len(ca)}")
print(f"spurious proj {m_sp:+.3f} [{lo_sp:+.3f},{hi_sp:+.3f}] n={len(sp)}")
print(f"causal > spurious: AUC={auc:.3f}, p={p:.4g}")

# ----------------------------
# Plots
# ----------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for cat in ["causal", "anti-causal", "spurious"]:
    sub = meta[meta["category"] == cat]
    axes[0].hist(
        100 * sub["frac_global_energy"],
        bins=25,
        alpha=0.45,
        density=True,
        label=cat
    )
axes[0].set_title(f"{MODEL_KEY} · global-vector energy")
axes[0].set_xlabel("% delta_h energy in global mean direction")
axes[0].set_ylabel("density")
axes[0].legend()

for cat in ["causal", "anti-causal", "spurious"]:
    sub = meta[meta["category"] == cat]
    axes[1].hist(
        sub["proj_global"],
        bins=25,
        alpha=0.45,
        density=True,
        label=cat
    )
axes[1].set_title(f"{MODEL_KEY} · projection on global dose vector")
axes[1].set_xlabel("projection scalar")
axes[1].set_ylabel("density")
axes[1].legend()

for cat in ["causal", "anti-causal", "spurious"]:
    sub = meta[meta["category"] == cat]
    axes[2].hist(
        sub["cos_to_global"],
        bins=25,
        alpha=0.45,
        density=True,
        label=cat
    )
axes[2].set_title(f"{MODEL_KEY} · cosine to global dose vector")
axes[2].set_xlabel("cos(delta_h, global mean)")
axes[2].set_ylabel("density")
axes[2].legend()

plt.tight_layout()
plt.savefig(tag("stage10_local_manifold_diagnostics.png"), dpi=180)
plt.show()

# ----------------------------
# Summary + save
# ----------------------------

summary = {
    "model": MODEL_KEY,
    "layer": STAGE10_LAYER,
    "contrast": CONTRAST_NAME,
    "n_items": len(meta),
    "global_mean_norm": float(np.linalg.norm(global_mean)),
    "mean_frac_global_energy": float(np.mean(frac_global_energy)),
    "median_frac_global_energy": float(np.median(frac_global_energy)),
    "mean_cos_to_global": float(np.mean(cos_global)),
    "median_cos_to_global": float(np.median(cos_global)),
    "proj_auc_causal_gt_spurious": float(auc),
    "proj_p_causal_gt_spurious": float(p),
    "acc3_global_projection": float(global_acc),
    "acc3_linear_full_delta": float(linear_acc),
    "acc_direct_global_projection": float(global_acc_d),
    "acc_direct_linear_full_delta": float(linear_acc_d),
}

for k, v in knn_accs.items():
    summary[f"acc3_knn_k{k}"] = float(v)

for k, v in knn_accs_d.items():
    summary[f"acc_direct_knn_k{k}"] = float(v)

if len(yb) >= 50:
    summary["r2_behav_global_projection"] = float(r2_global.mean())
    summary["r2_behav_linear_full_delta"] = float(r2_linear.mean())
    for k in K_LIST:
        knnr = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k, metric="euclidean"))
        r2_knn = cross_val_score(knnr, Xd, yb, cv=kf, scoring="r2")
        summary[f"r2_behav_knn_k{k}"] = float(r2_knn.mean())

summary_df = pd.DataFrame([summary])

meta.to_csv(tag("stage10_items_with_dose_geometry.csv"), index=False)
local_df.to_csv(tag("stage10_local_neighbor_consistency.csv"), index=False)
summary_df.to_csv(tag("stage10_summary.csv"), index=False)

print("\nSaved:")
print(tag("stage10_items_with_dose_geometry.csv"))
print(tag("stage10_local_neighbor_consistency.csv"))
print(tag("stage10_summary.csv"))
print(tag("stage10_local_manifold_diagnostics.png"))

print("\nFINAL INTERPRETATION TEMPLATE:")
print("If global energy is low but category prediction/local neighbours are high:")
print("  The dose signal is geometrically structured but heterogeneous — not one global knob.")
print("If global projection AUC causal>spurious is high:")
print("  A shared dose direction still captures a coarse causal-vs-spurious axis.")
print("If behavioural R² is low:")
print("  Hidden dose geometry is not directly read out as output probability.")

## §16 · Cleanup — free GPU (run LAST)

In [ ]:
del model; gc.collect()

if DEVICE=='cuda': torch.cuda.empty_cache(); print('GPU freed.')
print(f'{MODEL_KEY} complete. All outputs in {OUTDIR}/ (date-stamped {RUN_DATE}).')
print('Next model: set MODEL_KEY in §3 (size-ordered RUN_ORDER), restart runtime, re-run from §2.')
